# Fused residual add + RMSNorm kernel

In a Llama-style decoder, every sub-layer ends with a residual add and the next one starts with RMSNorm:

$$h = x + r, \qquad y_{i,j} = \frac{h_{i,j}}{\sqrt{\frac1D \sum_{k} h_{i,k}^2 + \varepsilon}}\; g_j$$

Inference engines fuse the two into one kernel that reads `x` and `r` once and writes both `h` (the new residual stream) and `y` (the input to the next attention or MLP block).

## Task
Write `fused_add_rmsnorm(x, r, g, eps)` returning the tuple `(h, y)`.

- `x`, `r`: `float32` `(rows, D)`; `g`: `float32` `(D,)`; `eps`: Python float. `rows, D >= 1`; `D` may be larger or smaller than the block.
- `h` and `y`: `float32` `(rows, D)`.

## Mapping
- One block of **64 threads per row**. Each thread handles columns `tid, tid + 64, ...`: it computes $h$, writes it, and accumulates $\sum h^2$ locally.
- Combine the 64 partial sums with a tree reduction in shared memory, then every thread scales its own columns.
- Accumulate in `float32`. Compared with relative tolerance $10^{-5}$, absolute $10^{-6}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda, float32

TPB = 64

def fused_add_rmsnorm(x, r, g, eps):
    """x, r: float32 (rows, D); g: float32 (D,). Returns (h, y): h = x + r, y = RMSNorm(h) * g."""
    raise NotImplementedError

@cuda.jit
def add_rmsnorm_kernel(x, r, g, eps, h, y):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 3.0]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 1.0]], "dtype": "float32"}, {"__nd__": [1.0, 1.0, 1.0], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[1.0, 2.0, 4.0]], "dtype": "float32"}, {"__nd__": [[0.37796443700790405, 0.7559288740158081, 1.5118577480316162]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[3.0, -4.0], [1.0, 1.0]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0], [1.0, -1.0]], "dtype": "float32"}, {"__nd__": [2.0, 0.5], "dtype": "float32"}, 1e-05], "e": {"__tup__": [{"__nd__": [[3.0, -4.0], [2.0, 0.0]], "dtype": "float32"}, {"__nd__": [[1.6970555782318115, -0.5656852126121521], [2.8284201622009277, 0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [0.695519745349884, -0.9794741868972778, -1.5734903812408447, -2.9249706268310547], "dtype": "float32"}, 1e-05], "e": {"__tup__": [{"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [[0.0, -0.0, -0.0, -0.0], [0.0, -0.0, -0.0, -0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.35323217511177063]], "dtype": "float32"}, {"__nd__": [[1.2476063966751099]], "dtype": "float32"}, {"__nd__": [0.03307262435555458], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[0.8943742513656616]], "dtype": "float32"}, {"__nd__": [[0.03307260200381279]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.5118440389633179, 1.023238182067871, -0.8821458220481873, 2.6522867679595947, -0.8769082427024841, 0.373553067445755, 2.7395808696746826, -0.11425241082906723, 0.11429451406002045, -0.511879563331604, 0.3345264792442322, -2.1266837120056152, -0.6468087434768677, 1.6940932273864746, 0.21132613718509674, -0.24039851129055023, 0.3729285001754761, 0.41460680961608887, 0.1521032601594925, 0.5785878896713257, -1.6595968008041382, 0.5912553071975708, -0.6735267639160156, 0.7858026623725891, -0.02124669775366783, -0.9981828331947327, -0.07213099300861359, -1.0070801973342896, 1.0127317905426025, -0.02859826199710369, 0.8530917167663574, -1.115056037902832, 1.6194173097610474, -0.3010613024234772, 1.3652658462524414, 0.8540513515472412, -1.1009451150894165, 0.323154479265213, -0.5424503087997437, 0.4093644917011261, 1.4324785470962524, 0.688670814037323, 1.0679359436035156, -1.1715694665908813, 0.20394518971443176, 0.3590717613697052, 1.088897705078125, -0.6194233298301697, 0.568786084651947, 0.42947131395339966, -1.5243183374404907, -1.0397233963012695, 0.44467079639434814, -1.1765680313110352, -1.2269164323806763, 0.8952400088310242, 0.2855663001537323, -0.23515623807907104, 0.10549359023571014, -0.4045258164405823, -0.5369294285774231, -0.4605884253978729, 0.3860619068145752, 1.3213037252426147], [-0.470756858587265, -0.0686572790145874, -0.21827809512615204, -1.2600277662277222, 0.25194138288497925, 0.8667653799057007, -0.84368497133255, 0.8330973386764526, -0.5716676115989685, 1.1513005495071411, 1.2158656120300293, -0.38307300209999084, -2.0009403228759766, -0.347988486289978, -1.3912076950073242, -1.3147207498550415, -0.2528270483016968, 1.3604499101638794, -0.09210652858018875, -0.21958014369010925, -1.8504993915557861, -0.6061244606971741, -1.3950831890106201, 0.5203277468681335, -0.5282455682754517, -0.04390937462449074, -0.016708428040146828, -0.18777482211589813, 0.9525678157806396, 0.4867924749851227, -0.3196272850036621, 0.15375912189483643, 0.5527752041816711, 1.3713452816009521, 0.12982802093029022, 1.4476189613342285, -0.5941755175590515, -0.23383170366287231, -1.487204909324646, 0.8218958973884583, -1.0326086282730103, -1.032463788986206, 0.8420940637588501, -0.5326586365699768, -0.7531147599220276, 0.953045666217804, -0.1843002885580063, 0.92033851146698, -0.4026825726032257, 0.7266659140586853, 1.0785411596298218, 0.0822918638586998, 1.7989991903305054, -0.39614051580429077, -0.21173031628131866, 0.07937099784612656, 1.0055204629898071, -0.2875935733318329, -0.4365914762020111, 0.25373736023902893, 1.249847173690796, -0.4165654480457306, -1.2776522636413574, -0.1308514028787613], [0.6237631440162659, -0.5427918434143066, 0.12424713373184204, -1.8689870834350586, 0.021277643740177155, 0.10273796319961548, 0.622498631477356, 0.309274286031723, 0.7717776298522949, -0.9943755865097046, 0.6602886319160461, -1.6512569189071655, -0.5968542098999023, 1.105857491493225, 0.8511833548545837, -1.8530139923095703, 0.588417649269104, -0.43080228567123413, -0.777921199798584, 0.20445320010185242, 0.12890809774398804, 1.394322156906128, -1.441975474357605, -0.428376168012619, -0.1425452083349228, -1.840631365776062, 1.1730940341949463, 1.7560858726501465, -0.6251199245452881, 0.3815532326698303, -0.26910629868507385, 0.2704012393951416, -0.6755243539810181, 0.8315537571907043, -1.1656098365783691, 1.363389015197754, 0.3563612103462219, 2.4247384071350098, -1.6513099670410156, -0.0955185741186142, -1.2855499982833862, -0.13992412388324738, 0.43511420488357544, -0.9195142388343811, -1.43507719039917, -1.7337679862976074, 0.22085289657115936, 1.2766908407211304, 1.171882152557373, 0.03375359997153282, -0.9616653919219971, -0.3632969260215759, -1.5078907012939453, -1.5763462781906128, 1.0659445524215698, -0.2781487703323364, 0.3144415020942688, -0.27350568771362305, 1.324920415878296, 1.4375256299972534, -0.7761577367782593, -0.36609378457069397, -0.6752188205718994, 1.3177608251571655], [1.2466017007827759, -0.2902878224849701, 2.2993927001953125, 0.026407383382320404, -1.2159340381622314, -2.033325672149658, -0.14375007152557373, 0.3213199973106384, -0.5866035223007202, 0.3630955219268799, 1.1950846910476685, -0.05982354283332825, 0.8428941369056702, 1.6645750999450684, 0.49771592020988464, -0.580569326877594, 0.4318041205406189, 0.5366910099983215, 2.1374528408050537, -0.43456894159317017, -1.55038321018219, 0.29880204796791077, 0.32579416036605835, -0.41567906737327576, 0.5647276043891907, 0.3182239830493927, 1.2110095024108887, -0.8351099491119385, -0.7024489045143127, -0.022564733400940895, -0.5077870488166809, 1.2710858583450317, 0.5791211128234863, -0.022218596190214157, 1.5277221202850342, 0.5015889406204224, 0.9994900822639465, 0.8050143718719482, -0.1474115401506424, -0.14031855762004852, 0.007356193847954273, -1.0909183025360107, -0.3449326753616333, 0.8542757630348206, 0.3097282946109772, -0.9438029527664185, 2.363450765609741, -1.0301765203475952, 0.6225975155830383, 0.7253857254981995, 1.2772561311721802, -0.8594166040420532, 0.024978281930088997, -0.1906512975692749, 1.1478232145309448, 0.3916192054748535, 0.4277717173099518, 0.551812469959259, 1.975608468055725, 0.6923210620880127, 0.9026769399642944, -0.906632661819458, 0.7231865525245667, -1.2160661220550537]], "dtype": "float32"}, {"__nd__": [[0.982259213924408, 0.1898108720779419, -0.8393492102622986, -1.0996456146240234, -0.041555918753147125, 0.17723314464092255, -0.5057705044746399, -0.26997315883636475, -0.2457820326089859, -0.46647849678993225, -0.1251077651977539, 0.33857330679893494, 1.3675733804702759, 0.7128583192825317, -0.4889460504055023, 1.6988322734832764, 1.267325758934021, -0.16692565381526947, 1.1095455884933472, -0.43814316391944885, -0.6564584374427795, -0.6212643384933472, 1.7212096452713013, 0.24452045559883118, 0.4235033690929413, 0.1771441400051117, -0.2898508608341217, 0.5998830199241638, 0.7955650091171265, 2.081915855407715, -0.7232158780097961, 1.6581454277038574, 0.7966205477714539, -1.5933194160461426, 1.1074728965759277, 1.7491307258605957, 1.2299939393997192, 0.38355889916419983, 0.29981374740600586, -1.8399895429611206, 0.46211445331573486, 0.3205260932445526, -0.7894413471221924, 0.3378620743751526, -0.5229359269142151, 2.3901724815368652, -1.0838655233383179, -0.7711196541786194, -2.0358774662017822, -0.2848977744579315, 0.22182650864124298, 0.11025763303041458, -0.5454539656639099, 0.9655859470367432, 0.32548418641090393, 1.2717607021331787, -0.029405107721686363, -1.6777589321136475, -0.06898269057273865, -0.9357827305793762, 0.8967223167419434, -0.5404817461967468, -0.1825566291809082, 0.03930474445223808], [-1.465519905090332, -0.336754709482193, 0.9182933568954468, -0.5262565016746521, -0.12514714896678925, 0.20234335958957672, -0.6383854150772095, -0.26161620020866394, 0.4681985676288605, -0.3501468002796173, 0.07893265038728714, -0.1181558147072792, -1.0222218036651611, -1.2724182605743408, 1.9966334104537964, 1.3938255310058594, -0.10336301475763321, 0.9643586277961731, 1.120504379272461, 0.9754855036735535, -0.3473757207393646, -1.4149301052093506, -1.2424442768096924, -0.2221895456314087, 1.3634353876113892, 1.5748670101165771, -0.2205970287322998, -0.31365880370140076, 1.2203428745269775, 2.076249837875366, -0.9003219604492188, 0.49133428931236267, 0.16811858117580414, -0.2962515354156494, -0.29284992814064026, -0.30683884024620056, 0.021522073075175285, -1.5092263221740723, 0.02463652193546295, 0.8125417232513428, 0.7807870507240295, 0.8181502223014832, 0.3587220311164856, 0.22334381937980652, 1.9701508283615112, -0.14686346054077148, 0.19122514128684998, 1.2753163576126099, 0.07038936018943787, -1.257455587387085, -0.5177369713783264, -0.39795172214508057, -0.12267448753118515, -0.4170135259628296, 1.0200775861740112, 0.736977756023407, 0.5167306661605835, -0.3662910461425781, 1.0564841032028198, 1.0222004652023315, 2.218618631362915, -0.5680578351020813, -0.06938859075307846, 1.4178773164749146], [-1.0361781120300293, 0.30606675148010254, 0.2626188099384308, 0.09904905408620834, -0.8959676027297974, 0.08469878882169724, -1.434892177581787, 1.2310295104980469, -0.44106525182724, 0.1979539394378662, 0.026265595108270645, 0.2265380620956421, 0.1607533097267151, 0.4572267234325409, -0.3268827199935913, 0.06456612050533295, -1.0710463523864746, -1.0101103782653809, -0.8174746632575989, -0.9785277247428894, -1.3395545482635498, -0.6001724004745483, 0.646896481513977, 0.43293920159339905, -0.39237648248672485, 0.3348243236541748, 0.38652193546295166, 2.6831471920013428, -1.5563621520996094, 0.27384254336357117, -1.2769238948822021, 1.2763147354125977, -0.18595650792121887, 0.47044116258621216, 0.23208962380886078, 0.727050244808197, -2.864046335220337, -1.1129781007766724, 0.11780927330255508, -1.283630132675171, 0.5851103067398071, 1.262794852256775, -1.2499042749404907, -0.38890570402145386, 0.6624733209609985, 1.841857671737671, 1.0418606996536255, 0.7625322937965393, -0.8273911476135254, 0.06878803670406342, 0.16540440917015076, -0.5614095330238342, 0.45051640272140503, -0.2706455886363983, -0.4004160761833191, 2.1519699096679688, -0.12795622646808624, 1.8971972465515137, -0.3459506332874298, -0.04031703993678093, -1.2039836645126343, -0.5404155254364014, 1.6045572757720947, 0.04260309040546417], [-1.9601185321807861, 1.4362291097640991, -0.8673393130302429, -0.4033053517341614, -0.02773703821003437, -1.859441876411438, 1.5441912412643433, 1.2869595289230347, -0.3229427635669708, 0.35771510004997253, -0.2817152142524719, -0.5895681381225586, -0.2302098274230957, -1.4323787689208984, -0.4032147526741028, -0.5566805005073547, 1.903481125831604, -1.457396388053894, 1.257964849472046, 0.7113572359085083, 0.14744272828102112, -0.598514199256897, 1.7983266115188599, 1.4505693912506104, -0.0429275706410408, -0.1966116577386856, 0.18776415288448334, 1.8169666528701782, -0.9046443104743958, 0.08274193853139877, 1.131484031677246, -1.3828637599945068, -0.0816090852022171, 1.109525203704834, -0.43519848585128784, -0.111329086124897, -1.5399606227874756, 0.37860289216041565, -1.1732627153396606, -0.3883366286754608, 0.33545759320259094, -0.5571152567863464, 0.8648982048034668, 0.7108448147773743, 0.4811137914657593, 0.9771313071250916, -0.16451658308506012, -2.148404359817505, 0.85665363073349, 0.7693972587585449, -0.4736260771751404, -0.8367120027542114, 1.1491787433624268, -0.4741206765174866, -0.38698166608810425, -0.44981661438941956, 2.318913698196411, -0.7250905632972717, -2.5936334133148193, -0.140427827835083, -1.0763031244277954, -1.800593376159668, 0.04995794966816902, -0.6180570125579834]], "dtype": "float32"}, {"__nd__": [0.7780670523643494, -0.43267080187797546, 1.0716017484664917, 0.8691127300262451, -0.08362890779972076, 0.6548380851745605, 1.6841366291046143, 0.5699825286865234, -1.5361604690551758, -0.036427952349185944, 0.08934922516345978, 1.1247453689575195, -0.24271710216999054, 1.0031585693359375, -0.24138405919075012, -0.08779150247573853, -0.35003700852394104, -0.10718552768230438, -0.2637229263782501, 0.21546907722949982, -0.880276620388031, -1.1618481874465942, -0.21432894468307495, 0.016339151188731194, -0.5332861542701721, 0.26279735565185547, 0.19284866750240326, 1.5317587852478027, -1.5890024900436401, 1.8173161745071411, -2.473118543624878, -1.914656162261963, 0.09224599599838257, -1.1235363483428955, -0.4859304428100586, -0.6345138549804688, -1.4966988563537598, -1.5609972476959229, -0.5906323194503784, 0.1419745236635208, 0.38293251395225525, -0.04587471857666969, -0.5383204221725464, 0.9671845436096191, -2.377720832824707, -0.18798872828483582, 1.0420974493026733, 1.3828473091125488, 0.9793737530708313, 1.12456214427948, 0.818193256855011, -0.07081969082355499, -2.4173169136047363, -0.6990925073623657, -0.4678410291671753, 1.373793125152588, -0.6164800524711609, -0.5379948019981384, 0.46735113859176636, -1.1439692974090576, -0.14690081775188446, 0.3703482151031494, -0.4976893663406372, -0.2365691363811493], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[1.494103193283081, 1.213049054145813, -1.7214950323104858, 1.5526411533355713, -0.918464183807373, 0.5507861971855164, 2.2338104248046875, -0.3842255771160126, -0.13148751854896545, -0.9783580303192139, 0.20941871404647827, -1.788110375404358, 0.7207646369934082, 2.406951427459717, -0.2776198983192444, 1.458433747291565, 1.640254259109497, 0.2476811558008194, 1.2616488933563232, 0.14044472575187683, -2.3160552978515625, -0.030009031295776367, 1.0476828813552856, 1.0303231477737427, 0.40225666761398315, -0.8210387229919434, -0.3619818687438965, -0.40719717741012573, 1.808296799659729, 2.0533175468444824, 0.12987583875656128, 0.5430893898010254, 2.4160377979278564, -1.8943806886672974, 2.472738742828369, 2.603182077407837, 0.12904882431030273, 0.7067133784294128, -0.2426365613937378, -1.430625081062317, 1.8945930004119873, 1.0091968774795532, 0.27849459648132324, -0.8337073922157288, -0.3189907371997833, 2.749244213104248, 0.005032181739807129, -1.390542984008789, -1.4670913219451904, 0.14457353949546814, -1.3024917840957642, -0.9294657707214355, -0.10078316926956177, -0.210982084274292, -0.9014322757720947, 2.1670007705688477, 0.2561611831188202, -1.9129152297973633, 0.0365108996629715, -1.3403085470199585, 0.35979288816452026, -1.0010701417922974, 0.203505277633667, 1.360608458518982], [-1.9362767934799194, -0.4054119884967804, 0.7000152468681335, -1.7862842082977295, 0.12679423391819, 1.0691087245941162, -1.4820704460144043, 0.5714811086654663, -0.10346904397010803, 0.8011537790298462, 1.2947982549667358, -0.5012288093566895, -3.0231621265411377, -1.6204067468643188, 0.6054257154464722, 0.07910478115081787, -0.3561900556087494, 2.3248085975646973, 1.028397798538208, 0.7559053897857666, -2.1978750228881836, -2.02105450630188, -2.6375274658203125, 0.29813820123672485, 0.8351898193359375, 1.530957579612732, -0.23730546236038208, -0.5014336109161377, 2.172910690307617, 2.563042402267456, -1.2199492454528809, 0.6450934410095215, 0.7208938002586365, 1.0750937461853027, -0.16302190721035004, 1.1407800912857056, -0.5726534724235535, -1.7430579662322998, -1.4625684022903442, 1.6344375610351562, -0.2518215775489807, -0.2143135666847229, 1.2008161544799805, -0.3093148171901703, 1.2170360088348389, 0.8061822056770325, 0.006924852728843689, 2.19565486907959, -0.33229321241378784, -0.5307896733283997, 0.5608041882514954, -0.31565985083580017, 1.676324725151062, -0.8131540417671204, 0.8083472847938538, 0.8163487315177917, 1.5222511291503906, -0.6538846492767334, 0.6198925971984863, 1.275937795639038, 3.468465805053711, -0.9846233129501343, -1.3470408916473389, 1.2870259284973145], [-0.4124149680137634, -0.2367250919342041, 0.3868659436702728, -1.7699379920959473, -0.8746899366378784, 0.1874367594718933, -0.8123935461044312, 1.5403038263320923, 0.33071237802505493, -0.7964216470718384, 0.6865542531013489, -1.4247188568115234, -0.43610090017318726, 1.5630842447280884, 0.5243006348609924, -1.7884478569030762, -0.4826287031173706, -1.4409127235412598, -1.595395803451538, -0.7740745544433594, -1.210646390914917, 0.7941497564315796, -0.7950789928436279, 0.004563033580780029, -0.5349217057228088, -1.5058070421218872, 1.559615969657898, 4.43923282623291, -2.1814820766448975, 0.6553957462310791, -1.5460301637649536, 1.5467159748077393, -0.8614808320999146, 1.3019949197769165, -0.9335201978683472, 2.0904393196105957, -2.5076851844787598, 1.3117603063583374, -1.5335006713867188, -1.3791487216949463, -0.7004396915435791, 1.122870683670044, -0.8147900700569153, -1.308419942855835, -0.7726038694381714, 0.10808968544006348, 1.2627135515213013, 2.0392231941223145, 0.34449100494384766, 0.10254164040088654, -0.7962609529495239, -0.9247064590454102, -1.0573742389678955, -1.8469918966293335, 0.6655284762382507, 1.8738211393356323, 0.18648527562618256, 1.6236915588378906, 0.9789698123931885, 1.397208571434021, -1.9801414012908936, -0.906509280204773, 0.9293384552001953, 1.3603639602661133], [-0.7135168313980103, 1.1459412574768066, 1.4320533275604248, -0.3768979609012604, -1.2436710596084595, -3.8927674293518066, 1.4004411697387695, 1.6082794666290283, -0.9095462560653687, 0.7208106517791748, 0.9133694767951965, -0.6493916511535645, 0.6126843094825745, 0.23219633102416992, 0.09450116753578186, -1.1372498273849487, 2.335285186767578, -0.9207053780555725, 3.3954176902770996, 0.27678829431533813, -1.4029405117034912, -0.2997121512889862, 2.1241207122802734, 1.0348902940750122, 0.5218000411987305, 0.12161232531070709, 1.3987736701965332, 0.9818567037582397, -1.6070932149887085, 0.06017720699310303, 0.6236969828605652, -0.1117779016494751, 0.4975120425224304, 1.0873066186904907, 1.0925235748291016, 0.39025986194610596, -0.540470540523529, 1.1836172342300415, -1.3206743001937866, -0.5286551713943481, 0.34281378984451294, -1.648033618927002, 0.5199655294418335, 1.5651205778121948, 0.7908420562744141, 0.033328354358673096, 2.1989340782165527, -3.1785807609558105, 1.4792511463165283, 1.4947829246520996, 0.8036300539970398, -1.6961286067962646, 1.1741570234298706, -0.6647719740867615, 0.7608415484428406, -0.05819740891456604, 2.74668550491333, -0.1732780933380127, -0.6180249452590942, 0.5518932342529297, -0.17362618446350098, -2.707226037979126, 0.7731444835662842, -1.834123134613037]], "dtype": "float32"}, {"__nd__": [[0.8962730169296265, -0.40464916825294495, -1.4222694635391235, 1.040374994277954, 0.05921904370188713, 0.2780735492706299, 2.900456428527832, -0.16884590685367584, 0.15572696924209595, 0.027477378025650978, 0.014426101930439472, -1.5505695343017578, -0.1348765790462494, 1.8615710735321045, 0.05166564881801605, -0.09871471673250198, -0.4426575303077698, -0.020467830821871758, -0.2565246522426605, 0.023330986499786377, 1.57184898853302, 0.026880916208028793, -0.17312245070934296, 0.012979132123291492, -0.16538886725902557, -0.16635173559188843, -0.053820304572582245, -0.4808812439441681, -2.215322732925415, 2.8769302368164062, -0.24763725697994232, -0.8016870021820068, 0.1718280166387558, 1.6409566402435303, -0.9263925552368164, -1.2734689712524414, -0.14891251921653748, -0.8505271077156067, 0.11048826575279236, -0.156595379114151, 0.5593464374542236, -0.03569373860955238, -0.11558469384908676, -0.6216782927513123, 0.5847653746604919, -0.39846283197402954, 0.004043033346533775, -1.4825226068496704, -1.1077667474746704, 0.125347301363945, -0.8216248750686646, 0.05074930936098099, 0.18782971799373627, 0.11371635645627975, 0.3251428008079529, 2.2952136993408203, -0.12175171077251434, 0.7934443950653076, 0.01315553579479456, 1.1821208000183105, -0.040749236941337585, -0.2858363687992096, -0.07808663696050644, -0.24816125631332397], [-1.1375024318695068, 0.13244086503982544, 0.5663811564445496, -1.1721805334091187, -0.008006150834262371, 0.5285957455635071, -1.884577751159668, 0.24594120681285858, 0.12000929564237595, -0.022035276517271996, 0.08734957128763199, -0.4256553649902344, 0.554025411605835, -1.2273304462432861, -0.11034110188484192, -0.005243522115051746, 0.09413771331310272, -0.18814437091350555, -0.20477499067783356, 0.12297596782445908, 1.4607975482940674, 1.7729450464248657, 0.42682090401649475, 0.0036780275404453278, -0.33628952503204346, 0.30377498269081116, -0.034553512930870056, -0.5799247622489929, -2.606959581375122, 3.516852378845215, 2.2780044078826904, -0.932569682598114, 0.050209589302539825, -0.912013590335846, 0.059811946004629135, -0.5465260744094849, 0.6471339464187622, 2.05438494682312, 0.6522306799888611, 0.1752050518989563, -0.0728086605668068, 0.007423196453601122, -0.4880734980106354, -0.22588008642196655, -2.1849029064178467, -0.11442815512418747, 0.005448623094707727, 2.292483329772949, -0.24571844935417175, -0.4506857097148895, 0.34644556045532227, 0.01687878742814064, -3.0595645904541016, 0.4292154312133789, -0.2855381667613983, 0.8467689156532288, -0.7085543870925903, 0.2656116187572479, 0.21873970329761505, -1.102076530456543, -0.3847064673900604, -0.2753266394138336, 0.5061823129653931, -0.22988638281822205], [-0.24137449264526367, 0.0770445242524147, 0.3118414282798767, -1.1571084260940552, 0.05502378195524216, 0.09232696890830994, -1.0291619300842285, 0.6604011654853821, -0.38214391469955444, 0.02182316593825817, 0.04614297300577164, -1.2053780555725098, 0.07962093502283096, 1.1794835329055786, -0.09519823640584946, 0.11810510605573654, 0.12707701325416565, 0.11617528647184372, 0.31648725271224976, -0.125460684299469, 0.8016344904899597, -0.6940515041351318, 0.12818317115306854, 5.608194260275923e-05, 0.21458059549331665, -0.29766666889190674, 0.22624251246452332, 5.114912509918213, 2.6074509620666504, 0.8959298729896545, 2.8760921955108643, -2.227620840072632, -0.059776850044727325, -1.100363850593567, 0.3412225544452667, -0.997742772102356, 2.823237180709839, -1.5402687788009644, 0.6813042163848877, -0.14728598296642303, -0.2017589956521988, -0.03874744474887848, 0.3299335539340973, -0.9519111514091492, 1.381839632987976, -0.015284666791558266, 0.9898127913475037, 2.1211867332458496, 0.2537851929664612, 0.08674084395170212, -0.49006223678588867, 0.04926038905978203, 1.922658920288086, 0.9712690711021423, -0.23420968651771545, 1.9363746643066406, -0.08647757023572922, -0.6570849418640137, 0.3441537618637085, -1.2023073434829712, 0.21880654990673065, -0.25253549218177795, -0.3479142189025879, -0.24207675457000732], [-0.4073364734649658, -0.3637910783290863, 1.1259645223617554, -0.24034327268600464, 0.07631221413612366, -1.870356798171997, 1.7305107116699219, 0.672597348690033, 1.025164246559143, -0.01926584355533123, 0.059878285974264145, -0.5359113812446594, -0.10911117494106293, 0.17090590298175812, -0.016737008467316628, 0.07325553894042969, -0.5997716784477234, 0.07240839302539825, -0.6570118069648743, 0.043758735060691833, 0.9061300158500671, 0.255497008562088, -0.33403509855270386, 0.012406689114868641, -0.20417216420173645, 0.02344934083521366, 0.1979229748249054, 1.103495478630066, 1.8736900091171265, 0.08024068176746368, -1.131750464439392, 0.15702863037586212, 0.033673107624053955, -0.8963367938995361, -0.38952648639678955, -0.1816883534193039, 0.5935243368148804, -1.355643630027771, 0.5723280310630798, -0.05506998673081398, 0.09631914645433426, 0.05547171086072922, -0.20537495613098145, 1.1106808185577393, -1.379693865776062, -0.0045970347709953785, 1.6813286542892456, -3.2250733375549316, 1.0629736185073853, 1.2333707809448242, 0.4824409484863281, 0.0881342887878418, -2.0825328826904297, 0.34098824858665466, -0.26117080450057983, -0.05866203084588051, -1.2423961162567139, 0.06839963793754578, -0.21192461252212524, -0.46323519945144653, 0.018714211881160736, -0.7356425523757935, -0.2823261618614197, 0.3183601200580597]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.9378083944320679, -1.4089927673339844, -0.42862915992736816, 0.9549633264541626, -0.7489762306213379, -1.0976351499557495, -0.9779293537139893, 0.46808069944381714, -1.601684331893921, -0.020109981298446655, -0.5456358790397644, -0.1165371760725975, -0.9265178442001343, -0.0955771803855896, 1.1665449142456055, -0.4661064147949219, -0.6703577041625977, -2.7298316955566406, 0.2923659384250641, -0.8189630508422852, 0.20231051743030548, -0.06730880588293076, -0.9626235961914062, -0.8156168460845947, -1.5099372863769531, 1.8078144788742065, 0.7557675242424011, 0.6611349582672119, 1.028045415878296, 1.2238883972167969, 0.7025846838951111, 0.8724191188812256, 2.3215534687042236, -1.840002417564392, 0.06727727502584457, -0.7429232001304626, -0.062110189348459244, -2.621852159500122, -1.186711311340332, 0.6975723505020142, -1.3565549850463867, 0.4462423026561737, -0.9254487156867981, -0.2657746970653534, -1.6016206741333008, -0.7074267268180847, -1.5928500890731812, 0.4574441909790039, -1.8186895847320557, 0.24448327720165253, -0.6206151843070984, -0.23869754374027252, 1.359108805656433, -0.31581738591194153, -0.6916343569755554, -0.03764471039175987, 0.6501632332801819, -0.5848644375801086, 1.6574770212173462, 0.14217624068260193, -0.3839366137981415, -0.6749065518379211, -0.170108363032341, 0.5449574589729309, 1.0393974781036377, 0.06812901794910431, -0.5890199542045593, -0.04568806663155556, -2.094144344329834, 1.1140161752700806, 0.8357539176940918, -0.18106810748577118, 0.7162753939628601, -0.9555338621139526, 1.836451768875122, -0.06807086616754532, -0.34352588653564453, 2.2083020210266113, -0.7727769017219543, 0.6280779242515564, -0.6039170026779175, -0.2923196852207184, -0.025417502969503403, -0.31273189187049866, 1.724039912223816, 1.364061951637268, -0.28932252526283264, 2.653055191040039, 0.2359744906425476, 0.101179338991642, 0.6774492263793945, -0.7508476972579956, 1.612385869026184, -0.6351020336151123, -0.9861252307891846, 1.312698483467102, -0.6653767824172974, -0.01385794673115015, -0.9754270911216736, -0.2249375581741333, 0.30789709091186523, -2.1907665729522705, 0.10945352911949158, -0.6278971433639526, 1.557196021080017, 2.172044038772583, 0.676404595375061, 0.7352904677391052, 1.1917668581008911, 1.1947412490844727, -1.2933590412139893, -1.1886078119277954, -0.37720274925231934, -1.283313512802124, -0.24386784434318542, 0.1819930523633957, -0.7227305173873901, 0.529170036315918, 1.492592692375183, 0.10764998942613602, 0.4847860634326935, 0.01746855303645134, -0.031049441546201706, -0.46501320600509644, -1.072483777999878, 0.597525417804718, 1.113789439201355, -0.010541650466620922, 0.13192157447338104, -1.330877423286438, -0.212544247508049, 0.32139238715171814, -0.07819341868162155, 0.07212144881486893, -0.9813310503959656, -1.2490931749343872, -0.24035656452178955, 1.4773257970809937, -0.3752252161502838, -1.4794312715530396, -0.6512056589126587, -0.20745135843753815, 1.1929595470428467, 1.0629173517227173, -0.7963200807571411, -1.0011365413665771, -0.2601618468761444, -0.018856031820178032, -0.9962247610092163, 0.9363133907318115, -0.4479416310787201, 0.23105348646640778, -0.21246331930160522, 0.4850236773490906, -0.1368219256401062, 0.2198350727558136, 1.1889081001281738, 0.20364463329315186, 1.3977049589157104, -1.277786135673523, -0.2250572293996811, -1.0629111528396606, -1.4387696981430054, -0.08063085377216339, 1.8346030712127686, 0.19602733850479126, 0.9435206055641174, -1.9327789545059204, 0.1785324662923813, -1.1857138872146606, 0.1669957935810089, 0.9532572627067566, -0.5389687418937683, 0.9996656179428101, -0.8481879234313965, 0.022585274651646614, -0.7099578976631165, -0.010600901208817959, -0.2546652853488922, -2.268141984939575, -0.4648091793060303, -0.6045658588409424, 1.5428681373596191, -2.582515001296997, -0.04531209543347359, 0.681496262550354, 1.745848536491394, 1.1569067239761353, -0.17288632690906525, 1.3000050783157349, -0.2542775571346283, -0.31307923793792725, -0.4664872884750366, -0.5012903213500977, 1.3711895942687988, -1.7524712085723877, -1.6657735109329224, 0.7935912609100342, 0.31456083059310913, 0.6954957842826843], [-0.4037773013114929, 0.4972366988658905, -0.3991956114768982, -0.6688748002052307, -0.6022185683250427, -0.03895116597414017, -0.11039403080940247, -0.5712307095527649, 1.378631591796875, 0.25560298562049866, -0.5313461422920227, -1.4865915775299072, -0.26888757944107056, -1.805404782295227, -1.080499529838562, -1.33883535861969, -1.0864956378936768, -0.04160543531179428, -3.4248619079589844, 1.2016136646270752, 0.18382488191127777, -0.6914722919464111, -0.8712969422340393, 0.4469239413738251, -0.38261041045188904, 1.4456864595413208, -0.41848066449165344, -1.1934436559677124, -1.6850557327270508, -0.37291282415390015, 0.7443552613258362, -1.482871174812317, 0.9779835939407349, 0.5547847747802734, -0.5854865908622742, 0.9956891536712646, -2.000558376312256, -2.4374547004699707, 0.0010952950688079, 0.1437794715166092, 0.10068660974502563, -0.31203219294548035, -1.7237080335617065, 0.27461665868759155, -0.6236423254013062, 0.6009752750396729, -0.05395273491740227, -1.4168416261672974, 0.3388943374156952, 0.5350136756896973, 0.5279734134674072, -0.9610094428062439, -1.2923716306686401, 0.5310443043708801, -0.31715691089630127, -0.3734881579875946, 0.7374219298362732, 0.7918720245361328, 0.2500893473625183, -0.4693525433540344, 1.2425439357757568, -1.2811908721923828, -0.2529918849468231, 0.07263683527708054, -1.6196545362472534, -0.962813138961792, 0.6240633726119995, -0.3287479877471924, -1.3000339269638062, 1.4150090217590332, -0.05596914142370224, -0.13595478236675262, 1.5845822095870972, -1.7058336734771729, -0.7702093124389648, -1.1493688821792603, 1.083016037940979, -1.0472731590270996, 2.11899995803833, 0.2471802681684494, 0.07680220156908035, 0.7481886744499207, -0.44339579343795776, -1.1320669651031494, -0.8319779634475708, -0.20219528675079346, -1.205661654472351, -0.09592028707265854, -0.8645432591438293, -0.4388495981693268, -0.3061888515949249, 1.3577020168304443, 2.0820651054382324, 1.6730847358703613, -0.28497257828712463, -0.7985181212425232, 0.970175564289093, 1.2624108791351318, 0.8894355893135071, -0.624445915222168, 0.28735801577568054, -0.28192856907844543, 1.024828553199768, 0.9226855039596558, 0.2220110148191452, -0.4078875780105591, 1.75740385055542, 0.207136332988739, 0.8458749651908875, 0.5826216340065002, 0.8181595206260681, 1.2064025402069092, -0.30490705370903015, -0.4017295837402344, 0.656375527381897, -3.165095090866089, -0.02187076210975647, -0.43141964077949524, 0.7738657593727112, 0.387970507144928, -0.1427532136440277, -1.1156210899353027, -0.1273394525051117, -0.27253109216690063, 0.11150185018777847, 0.2624591588973999, -0.664631724357605, 0.563224732875824, -0.4921778440475464, 0.44880637526512146, 0.3163784146308899, 1.2855844497680664, 0.021006401628255844, 1.628711462020874, -0.326862096786499, -0.8448825478553772, 0.669623076915741, -1.2092474699020386, 0.3920663297176361, 0.40898963809013367, 0.20791344344615936, 0.4281326234340668, 0.22449764609336853, 1.2991209030151367, -0.09706908464431763, -1.9221798181533813, -0.3378109335899353, 0.8594760298728943, 1.0157265663146973, 0.7077345848083496, 0.06262078881263733, -0.8913638591766357, 1.5847474336624146, -0.5520792603492737, -1.068454384803772, -2.396730899810791, 1.9183106422424316, 0.008171431720256805, -1.8295528888702393, 0.8732645511627197, 0.6714993119239807, -1.6622689962387085, 0.7854154109954834, 0.024661175906658173, 0.6787554025650024, 1.367431402206421, -0.4157227575778961, 2.172991991043091, -0.10518446564674377, -0.5953083634376526, 1.1753727197647095, -0.6730024218559265, -0.35618168115615845, 0.8077897429466248, -0.9011768698692322, -1.3751931190490723, 0.24104496836662292, -0.06721870601177216, -0.6278426647186279, 0.8688368201255798, -1.259070873260498, 0.6669791340827942, -1.1576850414276123, 0.3740290105342865, -1.3161636590957642, 0.5365521907806396, -0.17495128512382507, -0.5439648032188416, -0.06076040118932724, -0.27696922421455383, 0.681969404220581, -0.8066024780273438, 0.2788448631763458, -0.46808648109436035, -0.775178849697113, -0.671750545501709, -1.2709954977035522, 0.14798934757709503, -0.08930705487728119, 0.1696559488773346], [-1.7846018075942993, -0.6341149806976318, 1.294257640838623, 1.0117336511611938, -0.3099108338356018, -0.17724207043647766, 0.008977558463811874, 0.8776978254318237, 1.2867475748062134, 0.1323673129081726, 0.09609836339950562, -0.16678443551063538, 0.18747444450855255, 1.6201438903808594, -0.7862324118614197, 0.5090575814247131, -1.266075849533081, -1.2042206525802612, -1.7918530702590942, -0.7321520447731018, 1.3241180181503296, 1.4787176847457886, 0.6521911025047302, -0.17044445872306824, 0.9135810136795044, -1.185196876525879, -1.5476218461990356, -1.013757348060608, -0.36067116260528564, -1.9623088836669922, -0.9425378441810608, 1.5149898529052734, -0.3588300943374634, -0.5562474131584167, -0.8066188097000122, 0.5382997393608093, 0.476554811000824, -0.08741561323404312, -0.34316593408584595, -2.3037548065185547, -1.6464468240737915, -0.5999149680137634, -0.7615286707878113, -0.5136098861694336, -0.6767570376396179, 1.55446195602417, 0.9063865542411804, -1.2998312711715698, 0.5244559049606323, 0.16562800109386444, -0.2558952569961548, -0.46950581669807434, 0.13544325530529022, 1.6177914142608643, -0.7499352097511292, -0.38135451078414917, -0.33212512731552124, -1.8829033374786377, -0.20501305162906647, -1.025376319885254, 0.05150177329778671, 0.19564618170261383, 1.5837547779083252, -0.40759915113449097, -0.4616497755050659, 0.5420613288879395, 1.7857714891433716, 0.009168315678834915, 0.3239002823829651, 0.40673205256462097, 0.340177059173584, -0.6526259183883667, -0.9108392000198364, -0.8178539872169495, -0.5116909146308899, 0.15613992512226105, -0.996472954750061, -0.9043850302696228, -0.14364543557167053, -0.27817368507385254, -0.052382271736860275, -1.3746076822280884, 1.7063119411468506, -0.3582930862903595, -0.39335212111473083, -0.9879871010780334, 0.29137182235717773, 0.8426602482795715, -0.18606294691562653, -0.26077184081077576, -0.4038037359714508, 2.0249476432800293, 0.4133981168270111, -0.5664384365081787, 0.005818121135234833, 0.8843891620635986, 0.01147712953388691, 0.1575421392917633, 0.03489144891500473, -0.3153497874736786, 0.4171055257320404, 1.458674430847168, 0.3413947820663452, -0.4111647605895996, -0.926479697227478, -0.18883389234542847, -0.6517775058746338, -0.9451141357421875, 0.5118913054466248, 0.18845528364181519, 0.9454209804534912, 0.3627352714538574, -0.4787214398384094, 0.8110504746437073, 0.2963765263557434, 1.2662242650985718, -1.0429487228393555, -0.4249366819858551, -0.013673360459506512, 1.4355756044387817, 0.37732747197151184, -1.1814695596694946, -0.9514529705047607, 0.4521058201789856, -1.5713759660720825, -0.732067346572876, -0.20428824424743652, -0.41738876700401306, 0.2042410671710968, 0.6975436806678772, 0.18129627406597137, -1.3145920038223267, 1.1883000135421753, 0.9269649386405945, -0.8738405108451843, -0.4085279107093811, -1.3710596561431885, 1.267116665840149, -0.5329288244247437, 0.32738593220710754, 0.39099952578544617, 1.1523841619491577, 0.039037544280290604, -0.40434637665748596, 0.5147231817245483, 0.8749247789382935, -0.5725235939025879, -0.13623745739459991, -1.524759292602539, 1.2921451330184937, -0.6945185661315918, 0.8855481743812561, 0.779448390007019, 0.688632607460022, 0.8579670786857605, -0.8081333041191101, 0.1557905375957489, -0.7202978730201721, -0.16906531155109406, -0.7536611557006836, 2.251826524734497, 0.19351519644260406, 0.9391009211540222, -0.6480186581611633, 1.7152516841888428, -0.08920136094093323, -0.1628570407629013, -0.2668689787387848, -1.4763407707214355, -1.9325652122497559, 0.21102727949619293, -1.0138051509857178, -0.21949730813503265, 1.938181757926941, -0.2794436812400818, 1.9663952589035034, 0.3156645596027374, -1.050806999206543, 0.5693762898445129, 1.0783954858779907, 0.7626140713691711, -0.6604872345924377, -1.0378334522247314, 0.8367542624473572, 1.8204758167266846, 1.6995155811309814, -0.33943605422973633, -0.21299642324447632, 0.27893775701522827, 1.0820930004119873, 0.45092833042144775, 0.4460345506668091, 0.03548720106482506, -2.516761302947998, -1.1501768827438354, 0.1844651997089386, -1.4673197269439697, 2.0265040397644043, -0.7346568703651428, 0.6965457797050476]], "dtype": "float32"}, {"__nd__": [[0.9549564123153687, 1.1574039459228516, -1.0732945203781128, -0.13044658303260803, 1.5357266664505005, 0.1232021376490593, 0.8531568646430969, -2.322208881378174, 0.4302224814891815, 0.19933389127254486, -0.08012784272432327, -1.6865434646606445, -0.6799807548522949, -0.2258848249912262, -0.3344748616218567, -1.0411583185195923, 1.096447467803955, 0.9603222012519836, -0.3083049952983856, 1.4621437788009644, -1.1176598072052002, 0.33883050084114075, 1.6349575519561768, 0.6051207780838013, -0.10315999388694763, 0.5298241376876831, -0.06693766266107559, -0.4879375994205475, -0.7197703719139099, 0.055058274418115616, -0.10497333854436874, 0.346580445766449, -0.7820872664451599, 0.298482209444046, 0.9559280276298523, -0.5680865049362183, 1.575773000717163, -0.7015160322189331, -0.7039633393287659, -0.5931386351585388, -1.195420742034912, -1.5282344818115234, -1.1589734554290771, -0.2632655203342438, -0.8215693831443787, 1.4219263792037964, -0.22305648028850555, -0.4381699562072754, -0.589287281036377, 0.41642311215400696, 0.06574900448322296, 0.29687854647636414, 0.7931574583053589, 1.486628532409668, -0.8757657408714294, 0.6835289001464844, 0.05808079615235329, 0.2662363350391388, -0.5472374558448792, 0.6237837672233582, 0.6169202327728271, 0.3778057396411896, -0.7257566452026367, 0.6997721791267395, -0.9019356966018677, -1.4522422552108765, 0.8008195161819458, -0.09843721985816956, -1.512737512588501, -0.6554926037788391, 0.038325391709804535, 0.956326961517334, -0.19726522266864777, -0.7646734714508057, -0.9988222718238831, -1.2843477725982666, -0.09858284145593643, -0.7345246076583862, 0.617885947227478, -0.6675125956535339, 1.0418630838394165, -0.19320333003997803, 0.04892364889383316, -0.09008324891328812, -0.12060578167438507, 1.575217366218567, -0.6649194955825806, -0.47555142641067505, -0.21211712062358856, 0.6523004174232483, -0.9343706965446472, 1.410703182220459, -0.9097878336906433, -0.5188097953796387, 0.1108231246471405, 0.1714639514684677, -0.8757604360580444, 0.864364504814148, 1.7805671691894531, -0.004564851988106966, 0.7443400621414185, 0.7320883274078369, 2.513094425201416, 0.24708671867847443, 0.47383901476860046, -0.21006199717521667, 0.3973063826560974, -0.19418714940547943, -0.28211838006973267, -0.1658095419406891, 0.9307600259780884, -0.24774719774723053, 0.8768503069877625, -0.08243013918399811, 1.1866587400436401, -1.0975059270858765, -1.031468391418457, 0.6398742198944092, 1.4885026216506958, -0.3982095420360565, -1.9636178016662598, 1.115082025527954, 0.506852388381958, -0.6586720943450928, 1.3949782848358154, 1.3505761623382568, -0.5375359058380127, -0.32468023896217346, 0.6882101893424988, 0.6920458674430847, 1.3601871728897095, 0.5231713056564331, 1.9346909523010254, -0.3272768259048462, -0.8343454003334045, -0.5717964768409729, -0.718014121055603, -0.11324965953826904, -0.7897200584411621, 0.7696278691291809, 1.1491230726242065, 1.5691611766815186, -0.49065306782722473, -1.2333354949951172, -0.39171385765075684, 0.02515430748462677, -0.4858696460723877, 0.4192410707473755, 0.44207531213760376, 0.2914751172065735, -0.8817613124847412, 0.7554393410682678, -0.8918853402137756, -2.4143881797790527, -0.2626058757305145, 1.0011941194534302, -0.17032399773597717, -1.7289602756500244, -0.537608802318573, 0.8947976231575012, -0.06741126626729965, -0.350657194852829, -1.4116735458374023, -0.3124345541000366, -0.9024156928062439, 1.169274091720581, 0.037767887115478516, 0.525765597820282, -1.8023364543914795, -0.2705710530281067, 0.5209383368492126, 1.3720310926437378, 0.9916460514068604, -1.017077088356018, 2.523191213607788, -0.4039969742298126, -0.5254839062690735, -0.5384015440940857, -1.3009881973266602, -0.5643678307533264, -0.8873981833457947, -0.6820020079612732, -0.9062897562980652, -0.49992719292640686, 0.4078473150730133, -0.9193223714828491, -1.3791985511779785, 0.2900241017341614, -0.2891160249710083, 0.4032609462738037, -1.1294091939926147, -0.36015594005584717, -1.0432032346725464, -1.5251926183700562, -0.7345397472381592, -0.3851867914199829, 0.02249627374112606, -1.4047108888626099, 0.44528985023498535, 0.7491717338562012], [0.04315854609012604, -0.675582230091095, -1.075860619544983, 0.6690455079078674, -1.048193097114563, 0.9694762229919434, -2.013784885406494, -2.107769012451172, 0.14839284121990204, -1.4688584804534912, 0.693045973777771, -0.6796969175338745, -0.749999463558197, 0.0427076518535614, 0.787706196308136, 0.5534917712211609, -0.4169981777667999, -0.5525428652763367, 0.5444577932357788, -0.9638162851333618, -1.1217809915542603, -1.0750374794006348, 2.9071922302246094, 1.2394113540649414, 0.3238421678543091, -0.45876774191856384, 0.38782861828804016, -1.6651712656021118, -0.3502054512500763, 0.7747508883476257, 0.10373808443546295, -1.187670111656189, -0.14348994195461273, -0.2631209194660187, 1.2493518590927124, 0.7345926761627197, 0.10772301256656647, 2.0558130741119385, -0.4874892234802246, 1.1182332038879395, 0.4615807831287384, -0.07102692872285843, -0.4517746865749359, 0.5301358699798584, 1.482993245124817, -1.5166453123092651, 0.13697849214076996, 0.9265469908714294, -0.7134518623352051, -0.6491765975952148, 1.4055789709091187, -0.17884042859077454, -0.316595196723938, 0.46660926938056946, 0.3437372148036957, 0.6164270043373108, 0.6221715211868286, 1.460293173789978, 0.44820651412010193, 0.95432448387146, -0.6670404672622681, 0.30626675486564636, -0.41402488946914673, 1.470833659172058, 0.3921494781970978, 1.2461917400360107, -0.10773356258869171, -0.784173846244812, 0.8335568904876709, -0.05396042764186859, -1.3150032758712769, -0.19943839311599731, -0.6550827622413635, 1.4231524467468262, 0.00888907816261053, 0.7557916641235352, 0.5915993452072144, 1.2616536617279053, 0.5468382239341736, 0.5961804986000061, 0.49984338879585266, -1.176413655281067, 0.09941527247428894, -1.0770819187164307, -1.0686010122299194, 0.11096158623695374, -1.6601523160934448, -0.2276190221309662, 0.9076887965202332, -0.11479716002941132, 0.642120897769928, -1.0996390581130981, -0.14523333311080933, -2.624129295349121, -1.2114659547805786, -0.8177201151847839, 1.5126938819885254, 1.7318320274353027, -2.0222177505493164, -0.5657223463058472, -0.8843334913253784, 1.3598517179489136, 0.6627087593078613, -0.4933936893939972, 0.6220526099205017, 0.994801938533783, -0.769967257976532, 0.8089345097541809, 0.4143317937850952, 0.6089425683021545, -1.7395715713500977, -0.18455663323402405, -2.3864402770996094, 0.19421589374542236, -0.862757682800293, 1.1895809173583984, 1.2895163297653198, -1.7219140529632568, 1.403375506401062, -0.5064157843589783, 0.8768816590309143, 0.07883694767951965, -0.3954892158508301, -0.295774906873703, -0.5221997499465942, 1.326680064201355, 0.02922443114221096, -1.0263594388961792, 0.5865442752838135, -0.5189052820205688, 0.448581337928772, -0.8051607608795166, 0.2819088101387024, -0.952006459236145, -0.8243440389633179, 0.2580511271953583, 0.8012346029281616, 0.11621279269456863, -0.11390190571546555, 2.4982521533966064, -0.1434444785118103, -2.5250015258789062, 0.536112904548645, -0.5820102095603943, -1.4833000898361206, 0.9724781513214111, 0.7818886637687683, -0.4042128324508667, -0.5212436318397522, -0.04945560172200203, 0.9654809236526489, -0.9367407560348511, 1.2760512828826904, 0.2214871048927307, 0.3875875473022461, 1.499018669128418, -0.26779600977897644, 0.6206790804862976, 0.04352559521794319, -2.9088656902313232, -0.17079095542430878, 0.0152959655970335, -0.19019250571727753, 1.1583560705184937, -0.5339968204498291, 0.43962711095809937, 0.2809494137763977, 0.3782547116279602, -1.7322626113891602, -0.5135079622268677, -0.5400334000587463, 1.4824551343917847, -0.41813287138938904, 0.5222532749176025, 0.15272599458694458, -1.4523463249206543, -1.1995446681976318, 0.15184541046619415, 0.6062659025192261, 1.4338691234588623, -0.36526036262512207, -1.362237811088562, -0.18682055175304413, -1.4592187404632568, 1.2031205892562866, -0.44950947165489197, -0.17856404185295105, 1.6799651384353638, 0.2770752012729645, -1.3909755945205688, 0.2982414662837982, -0.7882778644561768, -1.3663150072097778, -0.9726760387420654, -1.5331097841262817, 0.3524840772151947, -0.9996773600578308, -0.5578832030296326, 0.848093569278717, -0.3547931909561157], [-0.625249445438385, -0.025053972378373146, -0.7081742286682129, -0.07812352478504181, -0.3450983166694641, 0.7144803404808044, -0.4833259880542755, -0.991115391254425, -0.5130521059036255, -2.2397117614746094, -0.2869507968425751, 0.6474730968475342, 1.770651936531067, 0.23828570544719696, 1.1002880334854126, -1.9249725341796875, 1.2011494636535645, 0.3185259997844696, 1.0688401460647583, -0.5932648181915283, 0.30294549465179443, 0.07156095653772354, -0.5252301096916199, -0.21485359966754913, 1.3399293422698975, -1.5912672281265259, -0.43836238980293274, 0.3446285128593445, 1.2491742372512817, -1.1823266744613647, -1.412227988243103, -2.0797245502471924, 0.16954758763313293, -0.30468693375587463, -0.1615840345621109, -0.22483402490615845, -1.8507623672485352, 0.9259944558143616, 0.7820407748222351, -1.42123544216156, -0.8276382684707642, -1.071333408355713, -0.8994983434677124, 1.6532026529312134, 0.810909628868103, -1.1509981155395508, 1.851229190826416, 0.3582768440246582, 0.07956469804048538, -0.8759334683418274, -0.8092746734619141, -0.2384541779756546, 2.732419967651367, -1.0976874828338623, -0.7706415057182312, 0.7962289452552795, 0.1156613752245903, -0.8305509686470032, 2.59429669380188, -0.289789080619812, -0.6442908644676208, -0.2516452968120575, 0.5033647418022156, 0.6079962849617004, 0.9094619750976562, 0.2534370422363281, -0.44995400309562683, 1.3619146347045898, -0.18188194930553436, -0.567412793636322, 0.7087148427963257, 1.1595218181610107, -0.46550658345222473, -0.47886794805526733, -0.18314065039157867, -0.7549620270729065, -0.38061684370040894, 0.7276911735534668, 0.6192785501480103, -2.3929667472839355, 1.4362651109695435, 1.3380736112594604, -1.0929419994354248, 1.4040465354919434, -1.3060495853424072, 1.4110807180404663, -0.5223925113677979, 0.5038272142410278, 0.22663290798664093, 0.6216964721679688, -1.2678871154785156, 1.5206763744354248, 0.7603219151496887, 0.20343491435050964, 1.2564141750335693, -0.02914908155798912, 2.0416698455810547, 1.4044249057769775, 2.0323312282562256, -0.6802598834037781, 0.2698017954826355, -1.7629516124725342, 0.24169254302978516, -0.7217321991920471, -0.7926285862922668, 0.8838611841201782, -0.10058175772428513, 1.0631778240203857, -1.1047639846801758, -0.7930783629417419, 0.24379374086856842, -2.256960391998291, -1.1443525552749634, -0.13795416057109833, -2.0143048763275146, 0.26995718479156494, -1.1660963296890259, 0.6100086569786072, -0.8561190962791443, 1.2999459505081177, -1.876514196395874, 1.0218011140823364, -1.639190912246704, 1.2131892442703247, -0.20060156285762787, -2.0351943969726562, 0.46587803959846497, -0.6888941526412964, -0.31120914220809937, -0.1721147894859314, 0.6959995031356812, -0.5165185332298279, 0.47909918427467346, -1.125589370727539, -0.6489766240119934, 0.3854379951953888, 1.1610538959503174, -0.04991220682859421, -0.32234272360801697, 0.37627607583999634, 0.49703311920166016, -0.9441209435462952, 2.8979694843292236, -0.44228142499923706, -1.2853742837905884, -0.052228644490242004, 0.3952687084674835, -0.07008016854524612, -0.6305420398712158, -0.07396699488162994, 0.5001551508903503, -0.6604222059249878, -0.8944593667984009, 0.12556231021881104, 0.11988819390535355, -1.1565167903900146, 2.3562848567962646, 0.9505341053009033, 1.6510473489761353, 1.0553362369537354, -1.434823751449585, 2.0518972873687744, -1.5978400707244873, -0.9359933137893677, 0.29198315739631653, -0.6126193404197693, 0.7172632217407227, 0.9218299984931946, -0.06444796174764633, 1.7394108772277832, -1.4724355936050415, 0.562791645526886, 0.7927530407905579, 1.9699039459228516, 0.9363260865211487, -1.5313401222229004, 0.6505265235900879, -0.2934352457523346, 0.21821700036525726, 0.5755182504653931, 1.266289234161377, -0.3694313168525696, 0.5591781139373779, 1.1329551935195923, 0.9374309778213501, 0.9619134664535522, -1.4386725425720215, 0.5220010876655579, 1.1559228897094727, -0.3276652693748474, 0.5080295205116272, -0.2962489128112793, 0.7519000172615051, 0.3864174783229828, -0.790242612361908, 0.6046540141105652, 0.5282244682312012, -1.228567361831665, -0.18694886565208435, -0.2824138402938843]], "dtype": "float32"}, {"__nd__": [-1.2020595073699951, 1.0817302465438843, -0.675071656703949, 0.11087880283594131, -2.0655367374420166, -0.14908669888973236, -1.3212774991989136, 0.36303383111953735, -0.290606826543808, 0.9215378761291504, 1.635128140449524, -1.7644877433776855, 0.06514614075422287, -0.1776205599308014, -0.5919191241264343, -0.8394852876663208, -0.12944014370441437, 0.527034342288971, 0.6104671359062195, 1.0600038766860962, -0.025362957268953323, 1.0339335203170776, 0.7037538886070251, 1.0717297792434692, 1.0268176794052124, 0.6930252909660339, 0.9824604988098145, -0.5917856097221375, -1.9417845010757446, -1.1642539501190186, 0.5205284357070923, 0.07641889899969101, -0.4503738582134247, 1.3692458868026733, 0.7994681000709534, 1.6843345165252686, -1.0330358743667603, -0.41028058528900146, 0.49531781673431396, 0.9758209586143494, 1.5121508836746216, 0.44956907629966736, 1.6135334968566895, 1.3607640266418457, 0.2845752537250519, 1.5054240226745605, 0.5288413166999817, 0.5952644348144531, -0.9474135041236877, -0.7055960297584534, -0.6219611167907715, 0.3839111626148224, -1.0558006763458252, -1.7254066467285156, -0.11227507889270782, 2.6080727577209473, 0.13301315903663635, -0.9646843075752258, -0.34236207604408264, 0.5568378567695618, 1.0933998823165894, 0.43576130270957947, 1.7996172904968262, 0.8549440503120422, -0.9253700375556946, 1.2011699676513672, -0.5514678955078125, 1.2334582805633545, 0.26680800318717957, -1.9678105115890503, -1.8885228633880615, -0.1406097561120987, 0.2091708481311798, 1.1775181293487549, 0.6383073925971985, 1.542974829673767, 0.8145424723625183, 0.1308286339044571, -0.9677348732948303, 1.4639956951141357, 1.1233323812484741, 1.0485801696777344, -0.7244481444358826, 1.127457857131958, -1.8386036157608032, 0.5857729911804199, 0.039530716836452484, 0.01587461307644844, -2.939340353012085, 1.9970036745071411, -1.1240384578704834, -0.8477500677108765, -0.47065839171409607, -0.6781821846961975, -1.9623956680297852, 0.6279347538948059, -0.28060880303382874, 0.12542635202407837, -1.6424602270126343, 0.29369431734085083, -0.022359078750014305, 1.3896276950836182, -0.7344051003456116, -2.067499876022339, 0.03434104472398758, 0.3478843569755554, 1.1368666887283325, 0.7405438423156738, -0.8729426860809326, 0.06474190950393677, -1.1363284587860107, 0.5540580153465271, 2.743844509124756, -1.0876742601394653, -1.486680507659912, -1.4702868461608887, -0.38715416193008423, 0.36440226435661316, -0.37472179532051086, -1.9903435707092285, 0.08467758446931839, 0.4731173813343048, 1.2574609518051147, 0.34967780113220215, -1.8253896236419678, 0.7478784322738647, -0.5821861624717712, -1.654916763305664, -0.33205148577690125, -0.10861139744520187, 0.07372477650642395, -1.096806287765503, -0.7180696725845337, -0.2292691171169281, 0.10280555486679077, 0.21751126646995544, -1.5388336181640625, 0.8044651746749878, -1.6183120012283325, 0.1255119889974594, -0.25460323691368103, -3.0619049072265625, -1.063551425933838, -0.3309381604194641, 0.05892004817724228, -0.4402955174446106, 0.2289799600839615, -0.32094040513038635, -0.5168639421463013, 0.6631502509117126, 3.6805803775787354, -0.6273130774497986, -0.49070078134536743, 0.3447582423686981, -0.944184422492981, 0.7169225215911865, -0.19432339072227478, 1.7336254119873047, -1.2345280647277832, -0.02752763405442238, -1.1061978340148926, 0.26879313588142395, -0.8435853123664856, -1.4147447347640991, -1.0492565631866455, -0.8340744972229004, -0.03123364970088005, -1.1188786029815674, 0.5175662636756897, -0.3684424161911011, -0.3482906222343445, -0.48041626811027527, -0.095659039914608, 1.6150506734848022, -1.0439847707748413, -0.118552066385746, 0.08116598427295685, 0.340527206659317, -0.6612957715988159, -2.4095852375030518, 0.5803418755531311, 0.5536674857139587, 0.8510012626647949, 0.49194666743278503, -1.1355146169662476, 1.3028630018234253, -0.35615062713623047, 0.40641337633132935, 2.0539588928222656, 1.2224533557891846, -0.8216651082038879, 0.12096595019102097, 1.4625734090805054, 0.968254029750824, -0.08670207858085632, -1.3408876657485962, -0.8825181722640991, -0.34282544255256653, 0.18942081928253174, 1.2694671154022217], "dtype": "float32"}, 1e-05], "e": {"__tup__": [{"__nd__": [[-0.9828519821166992, -0.2515888214111328, -1.501923680305481, 0.824516773223877, 0.7867504358291626, -0.9744330048561096, -0.12477248907089233, -1.854128122329712, -1.171461820602417, 0.1792239099740982, -0.6257637143135071, -1.803080677986145, -1.6064985990524292, -0.3214620053768158, 0.8320700526237488, -1.5072647333145142, 0.4260897636413574, -1.7695095539093018, -0.015939056873321533, 0.6431807279586792, -0.9153493046760559, 0.2715216875076294, 0.6723339557647705, -0.21049606800079346, -1.6130973100662231, 2.3376386165618896, 0.6888298392295837, 0.17319735884666443, 0.308275043964386, 1.2789466381072998, 0.5976113677024841, 1.2189996242523193, 1.539466142654419, -1.5415202379226685, 1.023205280303955, -1.3110096454620361, 1.5136628150939941, -3.3233680725097656, -1.8906745910644531, 0.10443371534347534, -2.551975727081299, -1.0819921493530273, -2.0844221115112305, -0.5290402173995972, -2.423190116882324, 0.7144996523857117, -1.8159065246582031, 0.019274234771728516, -2.4079768657684326, 0.6609063744544983, -0.5548661947250366, 0.058181002736091614, 2.152266263961792, 1.1708111763000488, -1.5674000978469849, 0.6458842158317566, 0.7082440257072449, -0.31862810254096985, 1.1102395057678223, 0.7659599781036377, 0.23298361897468567, -0.29710081219673157, -0.8958650231361389, 1.2447296380996704, 0.13746178150177002, -1.3841131925582886, 0.21179956197738647, -0.14412528276443481, -3.606881856918335, 0.45852357149124146, 0.8740792870521545, 0.7752588391304016, 0.5190101861953735, -1.7202073335647583, 0.837629497051239, -1.3524186611175537, -0.44210872054100037, 1.473777413368225, -0.15489095449447632, -0.03943467140197754, 0.437946081161499, -0.4855230152606964, 0.023506145924329758, -0.4028151333332062, 1.6034340858459473, 2.939279317855835, -0.9542419910430908, 2.177503824234009, 0.023857370018959045, 0.7534797787666321, -0.2569214701652527, 0.6598554849624634, 0.7025980353355408, -1.153911828994751, -0.8753020763397217, 1.484162449836731, -1.5411372184753418, 0.8505065441131592, 0.8051400780677795, -0.22950240969657898, 1.0522371530532837, -1.4586782455444336, 2.6225478649139404, -0.3808104395866394, 2.0310349464416504, 1.961982011795044, 1.0737109184265137, 0.5411033034324646, 0.9096484780311584, 1.028931736946106, -0.3625990152359009, -1.4363549947738647, 0.4996475577354431, -1.365743637084961, 0.9427908658981323, -0.9155128598213196, -1.7541989088058472, 1.1690442562103271, 2.981095314025879, -0.2905595600605011, -1.4788317680358887, 1.132550597190857, 0.4758029580116272, -1.123685359954834, 0.3224945068359375, 1.94810152053833, 0.5762535333633423, -0.33522188663482666, 0.820131778717041, -0.6388315558433533, 1.147642970085144, 0.8445637226104736, 1.8564975261688232, -0.25515538454055786, -1.8156764507293701, -1.8208897113800049, -0.9583706855773926, 1.3640761375427246, -1.1649452447891235, -0.7098034024238586, 0.49791741371154785, 1.3617098331451416, 0.7023065090179443, -0.1704181432723999, -1.188033938407898, -0.9759822487831116, -0.7460315227508545, 0.4003850519657135, -0.5541494488716125, 1.2277884483337402, -1.3297029733657837, 0.9864928126335144, -1.1043486595153809, -1.9293644428253174, -0.3994278013706207, 1.2210291624069214, 1.018584132194519, -1.5253156423568726, 0.8600961565971375, -0.38298851251602173, -0.29246848821640015, -1.413568377494812, -2.8504433631896973, -0.3930653929710388, 0.9321873784065247, 1.3653013706207275, 0.981288492679596, -1.4070134162902832, -1.623803973197937, -1.456284999847412, 0.687934160232544, 2.3252882957458496, 0.45267730951309204, -0.017411470413208008, 1.6750032901763916, -0.38141170144081116, -1.23544180393219, -0.549002468585968, -1.55565345287323, -2.832509756088257, -1.3522074222564697, -1.2865679264068604, 0.636578381061554, -3.082442283630371, 0.3625352084636688, -0.23782610893249512, 0.3666499853134155, 1.4469308853149414, -0.46200233697891235, 1.7032660245895386, -1.3836867809295654, -0.6732351779937744, -1.509690523147583, -2.0264830589294434, 0.6366498470306396, -2.13765811920166, -1.6432772874832153, -0.6111196279525757, 0.7598506808280945, 1.4446675777435303], [-0.3606187701225281, -0.17834553122520447, -1.4750561714172363, 0.00017070770263671875, -1.650411605834961, 0.9305250644683838, -2.124178886413574, -2.678999662399292, 1.5270243883132935, -1.213255524635315, 0.1616998314857483, -2.166288375854492, -1.0188870429992676, -1.7626971006393433, -0.292793333530426, -0.785343587398529, -1.5034937858581543, -0.5941482782363892, -2.880403995513916, 0.23779737949371338, -0.9379560947418213, -1.766509771347046, 2.035895347595215, 1.6863353252410889, -0.058768242597579956, 0.9869186878204346, -0.03065204620361328, -2.858614921569824, -2.0352611541748047, 0.4018380641937256, 0.8480933308601379, -2.670541286468506, 0.8344936370849609, 0.29166385531425476, 0.6638652682304382, 1.7302818298339844, -1.8928353786468506, -0.3816416263580322, -0.48639392852783203, 1.2620127201080322, 0.5622674226760864, -0.3830591142177582, -2.175482749938965, 0.80475252866745, 0.8593509197235107, -0.9156700372695923, 0.08302575349807739, -0.4902946352958679, -0.3745575249195099, -0.11416292190551758, 1.9335523843765259, -1.1398499011993408, -1.6089668273925781, 0.997653603553772, 0.02658030390739441, 0.2429388463497162, 1.359593391418457, 2.2521653175354004, 0.6982958316802979, 0.48497194051742554, 0.5755034685134888, -0.9749240875244141, -0.6670167446136475, 1.5434705018997192, -1.227505087852478, 0.28337860107421875, 0.516329824924469, -1.1129218339920044, -0.46647703647613525, 1.3610485792160034, -1.3709723949432373, -0.33539319038391113, 0.9294994473457336, -0.2826812267303467, -0.7613202333450317, -0.3935772180557251, 1.6746153831481934, 0.21438050270080566, 2.6658382415771484, 0.8433607816696167, 0.5766456127166748, -0.42822498083114624, -0.3439805209636688, -2.20914888381958, -1.9005789756774902, -0.09123370051383972, -2.865813970565796, -0.3235393166542053, 0.04314553737640381, -0.5536467432975769, 0.33593204617500305, 0.2580629587173462, 1.9368317127227783, -0.9510445594787598, -1.4964385032653809, -1.6162382364273071, 2.4828693866729736, 2.9942429065704346, -1.132782220840454, -1.1901682615280151, -0.5969754457473755, 1.0779231786727905, 1.6875373125076294, 0.42929181456565857, 0.8440636396408081, 0.5869143605232239, 0.9874365925788879, 1.01607084274292, 1.260206699371338, 1.1915642023086548, -0.9214120507240295, 1.0218459367752075, -2.691347360610962, -0.207513689994812, -0.206382155418396, -1.9755141735076904, 1.2676455974578857, -2.1533336639404297, 2.177241325378418, -0.11844527721405029, 0.734128475189209, -1.0367841720581055, -0.5228286981582642, -0.5683059692382812, -0.41069790720939636, 1.5891392230987549, -0.6354072690010071, -0.4631347060203552, 0.09436643123626709, -0.07009890675544739, 0.7649597525596619, 0.4804236888885498, 0.30291521549224854, 0.676705002784729, -1.151206135749817, -0.5868314504623413, 1.4708576202392578, -1.0930346250534058, 0.27816441655158997, 2.9072418212890625, 0.06446896493434906, -2.0968689918518066, 0.7606105804443359, 0.7171106934547424, -1.580369234085083, -0.9497016668319702, 0.444077730178833, 0.4552631974220276, 0.49448293447494507, 0.6582790017127991, 1.0281016826629639, -1.8281046152114868, 2.8607988357543945, -0.33059215545654297, -0.6808668375015259, -0.897712230682373, 1.6505146026611328, 0.628850519657135, -1.7860273122787476, -2.0356011390686035, 0.5007083415985107, -1.6469730138778687, 0.5952228903770447, 1.1830172538757324, 0.14475858211517334, 1.807058572769165, -0.1347733438014984, 2.5512466430664062, -1.8374470472335815, -1.108816385269165, 0.6353393197059631, 0.8094527125358582, -0.7743145227432251, 1.330043077468872, -0.7484508752822876, -2.8275394439697266, -0.9584996700286865, 0.084626704454422, -0.021576762199401855, 2.302706003189087, -1.6243312358856201, -0.6952586770057678, -1.3445055484771729, -1.085189700126648, -0.11304306983947754, 0.08704271912574768, -0.3535153269767761, 1.136000394821167, 0.21631479263305664, -1.6679447889328003, 0.9802109003067017, -1.5948803424835205, -1.0874701738357544, -1.4407625198364258, -2.30828857421875, -0.3192664682865143, -2.2706727981567383, -0.40989387035369873, 0.7587864995002747, -0.18513724207878113], [-2.409851312637329, -0.6591689586639404, 0.5860834121704102, 0.9336101412773132, -0.6550091505050659, 0.5372382402420044, -0.47434842586517334, -0.11341756582260132, 0.7736954689025879, -2.107344388961792, -0.19085243344306946, 0.4806886613368988, 1.958126425743103, 1.8584295511245728, 0.3140556216239929, -1.4159150123596191, -0.0649263858795166, -0.8856946229934692, -0.7230129241943359, -1.3254168033599854, 1.627063512802124, 1.550278663635254, 0.12696099281311035, -0.38529807329177856, 2.2535104751586914, -2.7764639854431152, -1.985984206199646, -0.6691288352012634, 0.8885030746459961, -3.1446356773376465, -2.3547658920288086, -0.564734697341919, -0.18928250670433044, -0.8609343767166138, -0.9682028293609619, 0.3134657144546509, -1.3742074966430664, 0.8385788202285767, 0.43887484073638916, -3.7249903678894043, -2.4740850925445557, -1.671248435974121, -1.661026954650879, 1.1395927667617798, 0.1341525912284851, 0.40346384048461914, 2.757615804672241, -0.9415544271469116, 0.6040205955505371, -0.7103054523468018, -1.0651699304580688, -0.7079600095748901, 2.867863178253174, 0.520103931427002, -1.5205767154693604, 0.41487443447113037, -0.21646374464035034, -2.713454246520996, 2.3892836570739746, -1.315165400505066, -0.5927891135215759, -0.055999115109443665, 2.0871195793151855, 0.20039713382720947, 0.44781219959259033, 0.7954983711242676, 1.3358174562454224, 1.3710829019546509, 0.14201833307743073, -0.16068074107170105, 1.0488919019699097, 0.506895899772644, -1.3763457536697388, -1.2967219352722168, -0.6948315501213074, -0.5988221168518066, -1.3770897388458252, -0.176693856716156, 0.4756331145763397, -2.671140432357788, 1.3838828802108765, -0.03653407096862793, 0.6133699417114258, 1.0457534790039062, -1.6994017362594604, 0.42309361696243286, -0.23102068901062012, 1.3464875221252441, 0.040569961071014404, 0.360924631357193, -1.671690821647644, 3.545624017715454, 1.1737200021743774, -0.36300352215766907, 1.2622323036193848, 0.8552401065826416, 2.053147077560425, 1.5619670152664185, 2.0672225952148438, -0.9956096410751343, 0.6869072914123535, -0.3042771816253662, 0.5830873250961304, -1.132896900177002, -1.7191083431243896, 0.6950272917747498, -0.7523592710494995, 0.11806368827819824, -0.592872679233551, -0.6046230792999268, 1.1892147064208984, -1.8942251205444336, -1.6230740547180176, 0.6730962991714478, -1.717928409576416, 1.5361814498901367, -2.209044933319092, 0.18507197499275208, -0.8697924613952637, 2.7355215549468994, -1.4991867542266846, -0.1596684455871582, -2.590643882751465, 1.665295124053955, -1.7719775438308716, -2.7672617435455322, 0.26158979535102844, -1.1062829494476318, -0.10696807503700256, 0.5254288911819458, 0.8772957921028137, -1.8311104774475098, 1.6673991680145264, -0.19862443208694458, -1.5228171348571777, -0.02308991551399231, -0.2100057601928711, 1.2172044515609741, -0.855271577835083, 0.7036620378494263, 0.8880326747894287, 0.20826321840286255, 2.937006950378418, -0.8466278314590454, -0.77065110206604, 0.8226961493492126, -0.17725488543510437, -0.20631763339042664, -2.155301332473755, 1.218178153038025, -0.19436341524124146, 0.2251259684562683, -0.11501097679138184, 0.814194917678833, 0.9778552651405334, -1.9646501541137695, 2.512075424194336, 0.2302362322807312, 1.4819819927215576, 0.30167508125305176, 0.8170027732849121, 2.245412588119507, -0.6587391495704651, -1.5840120315551758, 2.007234811782837, -0.7018207311630249, 0.5544061660766602, 0.6549609899520874, -1.5407887697219849, -0.19315433502197266, -1.2614083290100098, -0.4510135054588318, 0.573255717754364, 3.908085823059082, 0.6568824052810669, 0.435055136680603, 0.9661910533905029, -1.3442422151565552, 0.7875933051109314, 1.6539137363433838, 2.0289032459259033, -1.0299185514450073, -0.4786553382873535, 1.9697093963623047, 2.757906913757324, 2.661428928375244, -1.7781085968017578, 0.30900466442108154, 1.4348607063293457, 0.7544277310371399, 0.958957850933075, 0.14978563785552979, 0.7873871922492981, -2.1303439140319824, -1.9404194355010986, 0.7891192436218262, -0.9390952587127686, 0.7979366779327393, -0.9216057062149048, 0.41413193941116333]], "dtype": "float32"}, {"__nd__": [[0.9262786507606506, -0.21337221562862396, 0.7949234247207642, 0.07167630642652512, -1.2740823030471802, 0.1138986200094223, 0.12925294041633606, -0.5277332067489624, 0.2669079601764679, 0.12949013710021973, -0.8022130727767944, 2.494373083114624, -0.08205340802669525, 0.044766224920749664, -0.3861444890499115, 0.9920423626899719, -0.043241191655397415, -0.7311717867851257, -0.007628734223544598, 0.5345250368118286, 0.0182017982006073, 0.22010236978530884, 0.3709654211997986, -0.17687109112739563, -1.298618197441101, 1.2701470851898193, 0.5305845737457275, -0.08035876601934433, -0.469317764043808, -1.167421579360962, 0.243888258934021, 0.07303515076637268, -0.5435892939567566, -1.654848575592041, 0.6413446664810181, -1.7312581539154053, -1.2259482145309448, 1.0690228939056396, -0.7342235445976257, 0.07989846169948578, -3.025514602661133, -0.381371408700943, -2.636885166168213, -0.564415693283081, -0.5406450033187866, 0.8433123230934143, -0.7529158592224121, 0.008995279669761658, 1.788625717163086, -0.36561471223831177, 0.27056965231895447, 0.01751215197145939, -1.7815806865692139, -1.5838203430175781, 0.13797195255756378, 1.3206931352615356, 0.07385928928852081, 0.2409888058900833, -0.29800936579704285, 0.33439701795578003, 0.19972479343414307, -0.10150327533483505, -1.2640093564987183, 0.834334671497345, -0.09972980618476868, -1.3034777641296387, -0.09157414734363556, -0.13937735557556152, -0.7544984221458435, -0.7074121236801147, -1.294197678565979, -0.08546528220176697, 0.08511471003293991, -1.5880931615829468, 0.419188529253006, -1.6360536813735962, -0.2823387086391449, 0.1511688381433487, 0.11751952767372131, -0.04526323452591896, 0.3857060670852661, -0.3991526663303375, -0.0133510772138834, -0.35606861114501953, -2.3113555908203125, 1.3498880863189697, -0.02957473136484623, 0.027101270854473114, -0.05497940257191658, 1.1797175407409668, 0.22641704976558685, -0.4385753870010376, -0.2592628598213196, 0.6135452389717102, 1.3467038869857788, 0.7306737899780273, 0.33905503153800964, 0.08363614976406097, -1.0367974042892456, -0.0528457872569561, -0.01844569854438305, -1.5892256498336792, -1.5100338459014893, 0.617279589176178, 0.05468377470970154, 0.5351278185844421, 0.9570276141166687, 0.31416553258895874, -0.6225681304931641, 0.05222754925489426, 0.32304131984710693, -0.6239425539970398, 1.0748573541641235, 1.1646504402160645, -1.09890615940094, 1.0553441047668457, 0.5324641466140747, 0.33399471640586853, -0.8758150339126587, 0.45340970158576965, -0.09817814081907272, 0.4201013743877411, 0.46908247470855713, -0.3080635666847229, -0.46153590083122253, 1.1422736644744873, -0.26302865147590637, 0.43494677543640137, -0.2135092318058014, 0.054398804903030396, 0.0663357824087143, -0.7262563109397888, -1.0451738834381104, 0.04586462676525116, -0.14634659886360168, -0.3105224370956421, 1.1562527418136597, 0.8603466153144836, 1.4780712127685547, -0.06984750181436539, -0.0993914008140564, -3.268916606903076, -0.5856157541275024, 0.044217098504304886, -0.05488068237900734, 0.3369098901748657, -0.13393133878707886, -0.10074647516012192, 0.22455912828445435, 0.6383561491966248, -3.837059497833252, -0.4851832985877991, 0.42486459016799927, -0.5215026140213013, 0.29568052291870117, 0.6863186955451965, -0.15518493950366974, -2.073206663131714, -0.8324833512306213, 0.008265748620033264, 0.2536526918411255, -0.29789453744888306, 1.885250210762024, 0.43598392605781555, -0.7668536305427551, -0.8928139209747314, -0.024029627442359924, 1.2342660427093506, -0.6589115858078003, 0.4206719696521759, -0.1878521740436554, -0.875834584236145, -0.03395019844174385, -0.022046983242034912, -1.3709999322891235, 0.03545118495821953, -0.0786183550953865, -0.14657290279865265, 0.8065590858459473, 5.351078033447266, -0.6152545213699341, -0.5584821701049805, 0.42472678422927856, -1.1888868808746338, -0.3227531611919403, -0.24293263256549835, -0.10237949341535568, 0.4610450267791748, -0.743983805179596, 1.632459282875061, 0.8913745284080505, -0.06384950131177902, -1.7311439514160156, -1.538366675376892, -0.04327705502510071, 2.2472853660583496, 1.1370042562484741, 0.16425812244415283, 0.11284531652927399, 1.4378612041473389], [0.33619728684425354, -0.14962394535541534, 0.7722862958908081, 1.4679847481602337e-05, 2.643900156021118, -0.10759370774030685, 2.176732301712036, -0.7542922496795654, -0.34416893124580383, -0.867132306098938, 0.20506013929843903, 2.9645228385925293, -0.051479537039995193, 0.24282357096672058, 0.1344136744737625, 0.5113198757171631, 0.15093518793582916, -0.24285869300365448, -1.3637526035308838, 0.19549442827701569, 0.01845025271177292, -1.4165390729904175, 1.1112099885940552, 1.401680827140808, -0.04680108651518822, 0.5304571986198425, -0.02335578389465809, 1.3120185136795044, 3.0650746822357178, -0.36284294724464417, 0.34237968921661377, -0.15827777981758118, -0.29148492217063904, 0.3097304701805115, 0.4116242825984955, 2.2602949142456055, 1.5165200233459473, 0.12143855541944504, -0.18684951961040497, 0.9551107883453369, 0.6594136357307434, -0.13356174528598785, -2.722409963607788, 0.8493077158927917, 0.18966515362262726, -1.0690984725952148, 0.03405321389436722, -0.22635336220264435, 0.2752187252044678, 0.06247425824403763, -0.9326937198638916, -0.3393894135951996, 1.3174947500228882, -1.3350303173065186, -0.0023145319428294897, 0.4914016127586365, 0.14025668799877167, -1.685021162033081, -0.18541505932807922, 0.20944271981716156, 0.4880303740501404, -0.32948780059814453, -0.9309723973274231, 1.023424744606018, 0.8809649348258972, 0.2639923691749573, -0.22083470225334167, -1.0646553039550781, -0.09652704745531082, -2.077192783355713, 2.008033037185669, 0.03657544404268265, 0.15078918635845184, -0.258157342672348, -0.37689217925071716, -0.4709867537021637, 1.0579099655151367, 0.02175244130194187, -2.0008292198181152, 0.9575756788253784, 0.5023855566978455, -0.34825190901756287, 0.19326847791671753, -1.9317244291305542, 2.710153579711914, -0.041448090225458145, -0.0878622978925705, -0.0039833649061620235, -0.09835708141326904, -0.8574948310852051, -0.292854905128479, -0.16967326402664185, -0.7069969773292542, 0.5002269744873047, 2.277536392211914, -0.7871178984642029, -0.5403498411178589, 0.2912699580192566, 1.442982792854309, -0.27109643816947937, 0.010352143086493015, 1.16173255443573, -0.9611893892288208, -0.6883637309074402, 0.022480634972453117, 0.15835417807102203, 0.8706400990486145, 0.583572268486023, -0.8531932234764099, 0.05983052775263786, 0.8120404481887817, 0.4390971064567566, -5.72728967666626, 0.17505136132240295, 0.23796314001083374, 2.2526936531066895, -0.3806288242340088, -0.6085724234580994, -0.6327547430992126, 0.18283766508102417, 0.04821258783340454, -0.3804320991039276, -0.5098868608474731, -0.1541239619255066, 0.5814306735992432, 0.9217493534088135, 0.2869022488594055, 0.594433605670929, -0.0243020448833704, 0.005904818419367075, 0.043739307671785355, -0.40867137908935547, -0.1686970740556717, -0.12032748758792877, -0.09178877621889114, -0.0989954024553299, -1.7554247379302979, -0.6819633841514587, -0.34912723302841187, 0.2829999029636383, -0.012730179354548454, 4.979466915130615, -0.6273945569992065, -0.18405726552009583, -0.07221735268831253, 0.3243032991886139, 0.07886355370283127, -0.11332006752490997, -0.19821996986865997, 0.33856481313705444, 2.934757709503174, 0.8894166946411133, -1.0887393951416016, -0.0883948802947998, 0.4985846281051636, -0.4991479218006134, -0.248750701546669, 0.8455174565315247, 1.7100504636764526, 0.043459173291921616, -0.42957353591918945, -0.34334009885787964, -0.3894292116165161, -1.298042893409729, -0.11780016124248505, -1.1689527034759521, 0.0032647252082824707, -2.213886260986328, -0.7375656366348267, 0.31684666872024536, -0.17161980271339417, -0.3015984296798706, 0.05744646117091179, 1.665987491607666, 0.6060066223144531, 0.25997865200042725, -0.06033730506896973, 0.02235008403658867, 0.011066285893321037, -4.303290843963623, -0.731102705001831, -0.29854878783226013, -0.8873862624168396, -0.4140411913394928, 0.09955351054668427, 0.08795309066772461, 0.09764765948057175, 0.35806870460510254, 0.34458619356155396, -1.5813713073730469, -0.6246464252471924, -0.14962738752365112, -1.2335448265075684, -1.0819361209869385, 0.15521708130836487, 0.332021027803421, 1.554168462753296, 0.10898440331220627, 0.11147236078977585, -0.1822783648967743], [2.095146894454956, -0.515720009803772, -0.28615909814834595, 0.07487078011035919, 0.9785399436950684, -0.057930074632167816, 0.4533042311668396, -0.0297800675034523, -0.16262006759643555, -1.4045816659927368, -0.22570842504501343, -0.6134522557258606, 0.09226302802562714, -0.23874682188034058, -0.1344519853591919, 0.8597032427787781, 0.006078389007598162, -0.3376145660877228, -0.31923213601112366, -1.0161508321762085, -0.02984713390469551, 1.1593129634857178, 0.06462334096431732, -0.29866230487823486, 1.6735981702804565, -1.3916800022125244, -1.41120183467865, 0.28639960289001465, -1.247838020324707, 2.6479899883270264, -0.886523962020874, -0.03121357597410679, 0.06165693700313568, -0.8526087403297424, -0.5598424077033997, 0.38187098503112793, 1.0267530679702759, -0.24884167313575745, 0.15722545981407166, -2.6290180683135986, -2.7058768272399902, -0.543420135974884, -1.9384424686431885, 1.1215813159942627, 0.027611782774329185, 0.439300537109375, 1.0547691583633423, -0.40537187457084656, -0.4138944149017334, 0.36249294877052307, 0.4791598320007324, -0.19657927751541138, -2.189972400665283, -0.6490525603294373, 0.12347810715436935, 0.7825906276702881, -0.02082466520369053, 1.893241286277771, -0.5916319489479065, -0.5296724438667297, -0.46878910064697266, -0.017649324610829353, 2.7166004180908203, 0.12391602993011475, -0.2997157871723175, 0.6911017894744873, -0.5328017473220825, 1.223168969154358, 0.027405740693211555, 0.228689044713974, -1.4326868057250977, -0.05155045539140701, -0.2082221657037735, -1.1043652296066284, -0.32078027725219727, -0.6682745814323425, -0.8112865090370178, -0.016719471663236618, -0.3329099118709564, -2.8283607959747314, 1.124361276626587, -0.027707539498806, -0.32138684391975403, 0.8527621626853943, 2.2598648071289062, 0.179252028465271, -0.006605167407542467, 0.015459795482456684, -0.0862487331032753, 0.521307110786438, 1.3590497970581055, -2.1739962100982666, -0.39954763650894165, 0.178055539727211, -1.7915302515029907, 0.38841933012008667, -0.4166962802410126, 0.14169642329216003, -2.455728769302368, -0.21148663759231567, -0.011108368635177612, -0.30582016706466675, -0.3097187876701355, 1.6940809488296509, -0.04269873723387718, 0.17487792670726776, -0.618632972240448, 0.06323613971471786, 0.3743220865726471, -0.028311867266893387, -0.9773776531219482, -0.7590757608413696, -3.2210400104522705, -0.5295102000236511, 1.847230076789856, -1.6335891485214233, 0.6185669898986816, 0.04877755418419838, 0.23573434352874756, -3.9379160404205322, -0.09181686490774155, -0.05463692545890808, -2.3561367988586426, 0.42117008566856384, 2.3394408226013184, -1.4968541860580444, -0.11014910042285919, 1.3241604566574097, 0.025689631700515747, -0.04127505049109459, 0.04677974060177803, 1.4525890350341797, -0.865973174571991, 0.03293642774224281, -0.11323028802871704, -0.0036324728280305862, 0.23373359441757202, 0.708221435546875, 1.001070261001587, 0.06387745589017868, -0.16352762281894684, -0.4612140357494354, -2.2592365741729736, 0.20264582335948944, -0.032841213047504425, -0.2619883716106415, -0.02935580350458622, 0.047891560941934586, 0.8057169318199158, 0.5842800140380859, -0.5174031853675842, -0.10214290767908096, 0.040818244218826294, 0.20302116870880127, -0.6677742600440979, -1.0187220573425293, -0.3530661165714264, 0.28868696093559265, -1.3232507705688477, -0.006006294395774603, -0.6536640524864197, 0.4365285038948059, 0.40192103385925293, 1.620820164680481, -1.5232744216918945, 0.42337900400161743, -0.01252417266368866, -0.5300253629684448, -0.5767762064933777, 0.05147215723991394, 0.3177574574947357, 0.15671317279338837, -0.03966182470321655, 4.5650811195373535, -0.4959981441497803, -0.037303682416677475, 0.056719884276390076, -0.3310759365558624, -0.37670034170150757, -2.882394552230835, 0.8516154289245605, -0.41242992877960205, -0.2946126461029053, 0.7008392810821533, -2.265012502670288, 2.5079095363616943, 0.45802614092826843, 0.09083041548728943, 2.131570816040039, 0.667034387588501, -0.5698921084403992, 0.013104835525155067, 0.8329219222068787, -1.491891622543335, 0.12168117612600327, -0.7653027772903442, 0.5994204878807068, -0.19785168766975403, -0.12626156210899353, 0.3802405595779419]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[56.51165771484375, -88.56664276123047, -25.575777053833008, -47.305755615234375, -37.17009353637695, 24.448135375976562, -7.097465515136719, -59.01306915283203, 72.820556640625, -39.40397262573242, -59.12091064453125, -29.547679901123047, 88.77376556396484, 35.52862548828125, -35.73568344116211, 12.083398818969727, -50.35342025756836, 44.46867752075195, 15.491565704345703, -120.21905517578125, 6.006087303161621, 55.881690979003906, -76.13726806640625, -2.461456060409546, -26.78382682800293, 20.820756912231445, -77.19886779785156, 42.72364807128906, -16.94056510925293, 68.61898040771484, 38.175376892089844, -18.44491958618164, -60.335044860839844, -34.129886627197266, 66.39696502685547, 48.2577018737793, 17.08949851989746, 21.46969985961914, -6.743405818939209, -33.16348648071289, -15.45112133026123, -18.842687606811523, -39.23102569580078, 10.815850257873535, 104.96306610107422, -95.18789672851562, 27.43978500366211, 0.061736494302749634], [-60.33540344238281, 16.622190475463867, 110.9307632446289, -71.90660858154297, 5.893892288208008, -62.08613967895508, 54.165245056152344, 26.845914840698242, 5.5613017082214355, -10.431082725524902, 45.228050231933594, 51.77193832397461, -50.56635665893555, -78.81574249267578, -10.342854499816895, 43.894752502441406, -8.61842155456543, 84.82151794433594, 1.2760522365570068, -19.953798294067383, 19.20745849609375, -64.69505310058594, -44.859920501708984, -19.326078414916992, -29.987085342407227, 85.27500915527344, 50.96735382080078, 16.006610870361328, -16.82225799560547, 56.27588653564453, 63.490806579589844, -12.540035247802734, -68.15148162841797, 20.62200355529785, -10.294837951660156, -21.900405883789062, -0.8668360114097595, 49.298828125, -11.38036060333252, -6.0930328369140625, -27.218618392944336, -38.72071075439453, -51.1740608215332, 33.015777587890625, 3.182668924331665, -22.985700607299805, -16.308557510375977, 18.684511184692383], [30.833810806274414, 3.007734537124634, 7.849118232727051, 24.21377182006836, -40.41597366333008, -76.43970489501953, 102.59200286865234, 29.085338592529297, 70.1922378540039, -50.456695556640625, 8.238783836364746, 71.16024780273438, 81.58760833740234, 20.912187576293945, 56.356781005859375, 12.687593460083008, -47.34065246582031, -58.40016174316406, -74.94718170166016, -31.98048210144043, 54.493587493896484, -16.080631256103516, 97.16735076904297, 56.12108612060547, 42.74359893798828, -45.23188018798828, -6.5990471839904785, 40.38036346435547, -40.52214050292969, 45.26900863647461, 59.076202392578125, 28.793128967285156, -69.21484375, 17.571449279785156, -58.73884582519531, 76.81158447265625, -28.667306900024414, -26.271709442138672, 4.972991943359375, -26.66505241394043, 44.58950424194336, 16.91472053527832, 95.28834533691406, -74.49358367919922, 1.9461978673934937, 17.74828338623047, -8.01904582977295, -61.751564025878906], [-61.66588592529297, 19.62126922607422, -37.0262565612793, -93.34764099121094, -65.38517761230469, 9.781538963317871, 24.425146102905273, -5.7056803703308105, 32.18918228149414, 2.835953712463379, 11.864198684692383, -65.57083892822266, -38.066925048828125, 39.63773727416992, 12.728139877319336, 28.564565658569336, 50.50141906738281, -123.45975494384766, 31.122148513793945, 35.23750305175781, 6.1644392013549805, 14.004759788513184, -59.686248779296875, 29.832929611206055, 25.091848373413086, 11.1460542678833, 37.999046325683594, 75.132080078125, 10.754739761352539, -20.231168746948242, 5.771564483642578, 68.0723648071289, 41.690673828125, -2.036681890487671, 0.5934248566627502, 19.926475524902344, -57.082794189453125, -43.237884521484375, 24.63052749633789, 19.11058807373047, -22.300750732421875, -6.382787227630615, -15.629731178283691, 63.82573318481445, -13.21532917022705, 4.863238334655762, -14.908437728881836, -58.35223388671875], [-113.75516510009766, -39.61195755004883, 124.88285064697266, -33.88785934448242, 23.315006256103516, 3.8424596786499023, 67.66407775878906, -35.19017791748047, -73.43409729003906, -112.79608154296875, 63.6939582824707, 32.878936767578125, -119.42102813720703, 7.070191383361816, -45.32664108276367, -6.994324207305908, -40.14917755126953, 9.72055721282959, 79.18531799316406, 11.259430885314941, 16.934276580810547, 12.613179206848145, -13.585299491882324, -16.68000602722168, 46.01349639892578, 73.34837341308594, 33.187801361083984, 0.9698813557624817, 32.921913146972656, 43.16231155395508, 7.207457542419434, -38.87540054321289, 35.49039077758789, -65.28523254394531, 28.183496475219727, -45.90452575683594, -48.9466438293457, -10.775287628173828, -43.81772232055664, -3.446079730987549, 10.61291790008545, -12.895894050598145, 14.83956241607666, 58.81056213378906, -0.29185011982917786, -18.07057762145996, 17.3121280670166, 16.198848724365234], [-71.87165832519531, -10.244144439697266, -55.04711151123047, 63.53449249267578, -54.99533462524414, -32.24432373046875, 0.9452917575836182, -120.19962310791016, -98.4865951538086, -35.476993560791016, -19.115550994873047, -78.1900405883789, -8.616952896118164, 14.200882911682129, 15.658819198608398, 67.51429748535156, 24.96584129333496, -19.805700302124023, 57.71693420410156, 48.25107192993164, -20.7868595123291, -41.327171325683594, 21.434024810791016, 20.450000762939453, 72.72623443603516, 13.113456726074219, 53.597652435302734, 63.07079315185547, 18.29713249206543, 42.36442565917969, -64.06429290771484, -41.859737396240234, 5.170159339904785, -11.957345962524414, -3.9172346591949463, 12.240870475769043, -60.201316833496094, 62.695457458496094, 24.702327728271484, 41.93269348144531, -90.66992950439453, -48.90542221069336, -83.6937484741211, -54.8264274597168, 6.3986735343933105, 13.522884368896484, 48.55064010620117, -39.88821792602539]], "dtype": "float32"}, {"__nd__": [[0.8672512769699097, -0.8525230288505554, 0.49527332186698914, 0.9692299365997314, -1.0943424701690674, -0.40567922592163086, 0.47744080424308777, 0.5705331563949585, -1.2107722759246826, -1.6408721208572388, -1.3253693580627441, 0.4605898857116699, 0.5996690392494202, 0.19088008999824524, 0.6985644698143005, 0.6883147358894348, 0.19846151769161224, 1.2993510961532593, -2.01115083694458, -1.4520704746246338, -1.2522519826889038, 0.18911686539649963, -2.136031150817871, 1.663892149925232, 1.458245873451233, 1.5107619762420654, -1.8609040975570679, 0.45952466130256653, 0.03995863348245621, 1.2984702587127686, -0.2591931223869324, 0.145298033952713, -1.4776196479797363, 0.8680711984634399, 1.5494734048843384, 0.055063121020793915, -2.243439197540283, -0.7219159007072449, -1.2979836463928223, 0.2752225995063782, -0.3698921203613281, 0.6020774841308594, 0.4603951573371887, 0.37949809432029724, 1.1096627712249756, -1.1068772077560425, 1.1622955799102783, -0.3837115466594696], [-0.939195990562439, 0.9205194711685181, 0.5946357250213623, 1.1248059272766113, 1.3499960899353027, -0.2442023605108261, 0.9477295279502869, -1.2045822143554688, 0.1896504908800125, -0.2586800754070282, -0.8698887228965759, -1.5437687635421753, 0.4360451102256775, -1.7163820266723633, -0.5321885943412781, 1.6974185705184937, 1.6697043180465698, 0.5527955293655396, -0.30260637402534485, 1.6884448528289795, -0.11625374853610992, 0.264120489358902, 0.7110623121261597, -1.0432342290878296, -1.681543231010437, -1.9854618310928345, -1.503565788269043, -1.2610325813293457, -0.4745255708694458, 0.6792975068092346, -0.3578093647956848, -0.7998908758163452, 0.5848004221916199, -1.1062968969345093, 1.109795093536377, 0.4054979979991913, 1.40281081199646, 0.45637843012809753, 0.19817399978637695, 0.5685346126556396, 0.5278111696243286, 0.706599771976471, 0.33019718527793884, 0.9800925850868225, -0.7028300166130066, 1.3457316160202026, -0.3920994997024536, 2.2304794788360596], [-0.7237555980682373, 0.06517893075942993, 2.2840576171875, 1.185215950012207, -0.7392128109931946, 1.1960954666137695, 0.08622635900974274, -0.916920006275177, 1.7651008367538452, 0.36423513293266296, 2.299642324447632, 0.025152459740638733, 0.35223034024238586, 1.3354042768478394, -0.5133064985275269, 0.1674690991640091, -1.793609857559204, 1.1424227952957153, 1.5991408824920654, 1.300365686416626, 0.30321353673934937, -1.0288270711898804, 1.7443335056304932, -0.4380297064781189, 0.054890792816877365, 1.8043849468231201, 0.8755142092704773, 0.9283942580223083, 2.448209285736084, -0.08158731460571289, 1.1449284553527832, -0.995296835899353, -0.049819886684417725, -0.02058667503297329, 0.25670355558395386, 0.2058498114347458, -1.603914737701416, -0.7838248014450073, 0.7228024005889893, -1.2893636226654053, 1.1692215204238892, 1.484916090965271, -1.497464656829834, -1.363797903060913, 1.921024203300476, 1.039145588874817, -0.658668041229248, -0.5709580183029175], [-0.366690069437027, 0.3300354480743408, -1.2539414167404175, 1.4626517295837402, -0.8494011759757996, -0.6209009289741516, 0.25773197412490845, 0.957886815071106, 0.05065920576453209, 0.2897089123725891, 1.645843505859375, 0.10904867947101593, 0.003070634324103594, 1.0638773441314697, 0.7210102677345276, 0.046822287142276764, -0.4392148554325104, 0.7290760278701782, 0.6953728199005127, 0.8895077109336853, -0.4415169954299927, -1.4384982585906982, -1.132726788520813, 2.4613022804260254, -0.3794960379600525, -0.5824332237243652, -0.37721967697143555, -1.3095190525054932, -0.9352002143859863, 0.812117874622345, 1.0849241018295288, -0.8831517696380615, -0.1023111492395401, -0.23437455296516418, 0.38408827781677246, -0.30816543102264404, -0.286540687084198, -0.47448790073394775, -0.5383705496788025, -1.0657804012298584, 0.3100113868713379, 0.6989840865135193, -1.5023072957992554, 0.656570315361023, -0.8123242259025574, 0.8141112327575684, 0.43547701835632324, -2.514979124069214], [-1.818093180656433, 0.06508252769708633, 0.37753739953041077, 0.1069236695766449, -0.8529891967773438, 0.3110010325908661, -2.351732015609741, 0.9027701616287231, 0.3731006383895874, -0.5036928057670593, 0.7404350638389587, -1.3653907775878906, 0.43776774406433105, 0.591834545135498, -0.18211109936237335, 0.5656241774559021, -0.4396115243434906, 1.8284395933151245, -2.30592942237854, -0.14587463438510895, 0.4454452693462372, 0.9236815571784973, 1.687034249305725, -0.004636815749108791, -0.641495943069458, 0.17515546083450317, -0.4449317455291748, 0.13681432604789734, -0.5293704271316528, -1.7466537952423096, 1.2438983917236328, -1.0561479330062866, 0.7297033071517944, 0.9714516401290894, -0.244607612490654, 1.2103140354156494, -1.22198486328125, -0.7105389833450317, 0.1276146024465561, 1.041468858718872, -0.7722501754760742, -1.000504493713379, 0.4490565359592438, 0.6835565567016602, -1.1844725608825684, 1.015379786491394, 1.7770246267318726, 0.764015257358551], [0.9749891757965088, 1.4304993152618408, -0.7239980101585388, 0.6274664998054504, -0.6436716914176941, 0.28939250111579895, 0.13382168114185333, -0.3232007622718811, -1.2659766674041748, -1.0384464263916016, -1.7532472610473633, -1.0453463792800903, -0.7686207890510559, 0.9970507025718689, 0.5195732116699219, 0.4309380352497101, -0.9678317308425903, -0.8815190196037292, -0.3029346764087677, 0.9115657806396484, -1.6513667106628418, -1.0262699127197266, 0.32936719059944153, -0.33480212092399597, 0.963320255279541, 0.3989957571029663, -0.7444333434104919, -0.6315212249755859, -0.6742730736732483, -0.3832562565803528, 0.9431657791137695, -0.31503382325172424, -0.45532241463661194, 1.0420584678649902, 0.25470373034477234, 1.7597687244415283, 0.5180110335350037, -0.5223156809806824, 0.9756251573562622, -0.2302914261817932, 0.49345818161964417, 0.4396379590034485, 1.5206013917922974, 1.4737870693206787, 0.17996442317962646, 0.3884783685207367, -0.6917117238044739, -0.6462587118148804]], "dtype": "float32"}, {"__nd__": [0.3075573444366455, 0.3434235453605652, -1.0512207746505737, 0.37256020307540894, -0.39743098616600037, 0.9623034000396729, -1.1539751291275024, 0.8818644285202026, 0.005235011223703623, -0.8986320495605469, 0.07803010195493698, -0.45578673481941223, -1.4215255975723267, -1.4543606042861938, 0.893100380897522, -0.15178582072257996, 0.818103551864624, 1.5713931322097778, -0.30738943815231323, -0.41898664832115173, -1.5924071073532104, 0.230396568775177, -1.0090433359146118, -1.4247856140136719, -0.16585753858089447, 0.828746497631073, 0.2525880038738251, -1.6760330200195312, 2.1000053882598877, 1.1670705080032349, 0.41633161902427673, 1.225342869758606, -0.26362332701683044, 0.4854141175746918, 0.9616355299949646, 0.10233883559703827, 1.444764494895935, -0.5514836311340332, 0.3518955707550049, 0.22542843222618103, -1.2218255996704102, -0.9659097194671631, -1.2423051595687866, 1.4826440811157227, 0.23785492777824402, -1.3794944286346436, -0.4876480996608734, -1.0010197162628174], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[57.378910064697266, -89.4191665649414, -25.080503463745117, -46.336524963378906, -38.264434814453125, 24.042455673217773, -6.620024681091309, -58.442535400390625, 71.60978698730469, -41.04484558105469, -60.44628143310547, -29.08708953857422, 89.3734359741211, 35.719505310058594, -35.0371208190918, 12.771713256835938, -50.15496063232422, 45.768028259277344, 13.480415344238281, -121.67112731933594, 4.753835201263428, 56.07080841064453, -78.27330017089844, -0.797563910484314, -25.325580596923828, 22.331518173217773, -79.05976867675781, 43.18317413330078, -16.900606155395508, 69.91744995117188, 37.91618347167969, -18.29962158203125, -61.81266403198242, -33.26181411743164, 67.94644165039062, 48.31276321411133, 14.846059799194336, 20.747783660888672, -8.041389465332031, -32.88826370239258, -15.821013450622559, -18.240610122680664, -38.7706298828125, 11.195348739624023, 106.0727310180664, -96.2947769165039, 28.602081298828125, -0.32197505235671997], [-61.274600982666016, 17.542709350585938, 111.52539825439453, -70.78179931640625, 7.2438883781433105, -62.33034133911133, 55.11297607421875, 25.641332626342773, 5.750952243804932, -10.689763069152832, 44.35816192626953, 50.22816848754883, -50.13031005859375, -80.5321273803711, -10.875042915344238, 45.59217071533203, -6.94871711730957, 85.37431335449219, 0.9734458923339844, -18.26535415649414, 19.091205596923828, -64.4309310913086, -44.14885711669922, -20.369312286376953, -31.668628692626953, 83.28955078125, 49.46378707885742, 14.74557876586914, -17.296783447265625, 56.95518493652344, 63.13299560546875, -13.339925765991211, -67.56668090820312, 19.51570701599121, -9.185043334960938, -21.49490737915039, 0.5359748005867004, 49.75520706176758, -11.182186126708984, -5.524497985839844, -26.690807342529297, -38.01411056518555, -50.84386444091797, 33.99586868286133, 2.4798388481140137, -21.639968872070312, -16.70065689086914, 20.91499137878418], [30.110055923461914, 3.072913408279419, 10.13317584991455, 25.39898681640625, -41.15518569946289, -75.24360656738281, 102.67823028564453, 28.168418884277344, 71.95733642578125, -50.09246063232422, 10.538426399230957, 71.1854019165039, 81.9398422241211, 22.247591018676758, 55.843475341796875, 12.855062484741211, -49.13426208496094, -57.25774002075195, -73.34803771972656, -30.680116653442383, 54.79680252075195, -17.109458923339844, 98.91168212890625, 55.68305587768555, 42.79848861694336, -43.427494049072266, -5.7235331535339355, 41.30875778198242, -38.07393264770508, 45.18741989135742, 60.22113037109375, 27.797832489013672, -69.26466369628906, 17.55086326599121, -58.48214340209961, 77.0174331665039, -30.271221160888672, -27.05553436279297, 5.695794105529785, -27.954416275024414, 45.758724212646484, 18.39963722229004, 93.79087829589844, -75.85738372802734, 3.8672220706939697, 18.787429809570312, -8.677713394165039, -62.3225212097168], [-62.03257751464844, 19.951305389404297, -38.28019714355469, -91.8849868774414, -66.23458099365234, 9.160637855529785, 24.682878494262695, -4.747793674468994, 32.23984146118164, 3.1256625652313232, 13.510042190551758, -65.4617919921875, -38.0638542175293, 40.70161437988281, 13.449150085449219, 28.611387252807617, 50.06220245361328, -122.73068237304688, 31.817522048950195, 36.127010345458984, 5.722922325134277, 12.566261291503906, -60.81897735595703, 32.29423141479492, 24.712352752685547, 10.563621520996094, 37.621826171875, 73.82256317138672, 9.819540023803711, -19.419050216674805, 6.8564887046813965, 67.18921661376953, 41.58836364746094, -2.2710564136505127, 0.9775131344795227, 19.618310928344727, -57.36933517456055, -43.712371826171875, 24.0921573638916, 18.04480743408203, -21.990739822387695, -5.683803081512451, -17.132038116455078, 64.4822998046875, -14.027653694152832, 5.67734956741333, -14.472960472106934, -60.86721420288086], [-115.57325744628906, -39.546875, 125.26039123535156, -33.78093719482422, 22.462017059326172, 4.153460502624512, 65.31234741210938, -34.28740692138672, -73.06099700927734, -113.29977416992188, 64.43439483642578, 31.513545989990234, -118.98326110839844, 7.6620259284973145, -45.508750915527344, -6.428699970245361, -40.58879089355469, 11.548996925354004, 76.87938690185547, 11.113555908203125, 17.379722595214844, 13.536860466003418, -11.89826488494873, -16.684642791748047, 45.37200164794922, 73.52352905273438, 32.74287033081055, 1.1066956520080566, 32.39254379272461, 41.41565704345703, 8.451355934143066, -39.931549072265625, 36.2200927734375, -64.31378173828125, 27.938888549804688, -44.694210052490234, -50.16862869262695, -11.48582649230957, -43.69010925292969, -2.4046108722686768, 9.840667724609375, -13.896398544311523, 15.288619041442871, 59.494117736816406, -1.4763226509094238, -17.055198669433594, 19.089153289794922, 16.96286392211914], [-70.89666748046875, -8.813645362854004, -55.77111053466797, 64.16195678710938, -55.639007568359375, -31.954931259155273, 1.079113483428955, -120.5228271484375, -99.75257110595703, -36.51544189453125, -20.868797302246094, -79.23538970947266, -9.385573387145996, 15.197933197021484, 16.17839241027832, 67.94523620605469, 23.998008728027344, -20.687219619750977, 57.413997650146484, 49.162635803222656, -22.4382266998291, -42.35343933105469, 21.763391494750977, 20.115198135375977, 73.6895523071289, 13.512452125549316, 52.85321807861328, 62.43927001953125, 17.622859954833984, 41.981170654296875, -63.12112808227539, -42.17477035522461, 4.714837074279785, -10.915287017822266, -3.6625308990478516, 14.000638961791992, -59.68330764770508, 62.17314147949219, 25.677953720092773, 41.70240020751953, -90.17646789550781, -48.46578598022461, -82.17314910888672, -53.35264205932617, 6.578638076782227, 13.911362648010254, 47.85892868041992, -40.53447723388672]], "dtype": "float32"}, {"__nd__": [[0.34741267561912537, -0.6045440435409546, 0.5190359950065613, -0.33984994888305664, 0.2993810474872589, 0.45546823740005493, 0.15039150416851044, -1.0146076679229736, 0.007380015216767788, 0.7261189818382263, -0.09285365790128708, 0.26099321246147156, -2.501096248626709, -1.0226918458938599, -0.6160216331481934, -0.0381634496152401, -0.807773232460022, 1.41584050655365, -0.08157544583082199, 1.0035868883132935, -0.14902718365192413, 0.2543197572231293, 1.5548579692840576, 0.02237083949148655, 0.08269169926643372, 0.36434030532836914, -0.3931294083595276, -1.4248343706130981, -0.6986987590789795, 1.6063861846923828, 0.3107644021511078, -0.4414352476596832, 0.3207957148551941, -0.3178524971008301, 1.286306619644165, 0.09733512997627258, 0.4222557246685028, -0.22525352239608765, -0.055707305669784546, -0.14595431089401245, 0.3805491626262665, 0.3468511700630188, 0.9481966495513916, 0.32676973938941956, 0.4966874420642853, 2.615109920501709, -0.2745816111564636, 0.006345007102936506], [-0.42802155017852783, 0.1368314027786255, -2.6627275943756104, -0.5989314317703247, -0.06538716703653336, -1.3622932434082031, -1.4444727897644043, 0.5135721564292908, 0.0006837800028733909, 0.21817702054977417, 0.07861307263374329, -0.5199577808380127, 1.6185053586959839, 2.6601145267486572, -0.22059227526187897, -0.15717393159866333, -0.1291135847568512, 3.0469906330108643, -0.00679610064253211, 0.1738150268793106, -0.6904726624488831, -0.33715489506721497, 1.0117851495742798, 0.659151017665863, 0.11929535120725632, 1.5677299499511719, 0.28376519680023193, -0.5613108277320862, -0.8249826431274414, 1.5096956491470337, 0.596973180770874, -0.371253103017807, 0.40455302596092224, 0.2151571363210678, -0.20060914754867554, -0.04996145889163017, 0.01758735254406929, -0.6232039928436279, -0.08937160670757294, -0.028285283595323563, 0.7406789064407349, 0.8339507579803467, 1.4345836639404297, 1.144780158996582, 0.013396602123975754, 0.6780092716217041, 0.18496908247470856, -0.4755096733570099], [0.18257886171340942, 0.020806221291422844, -0.21001596748828888, 0.18656307458877563, 0.3224772810935974, -1.4275602102279663, -2.3360798358917236, 0.48975303769111633, 0.007426864933222532, 0.8874973654747009, 0.016212528571486473, -0.6396839618682861, -2.2964797019958496, -0.637922465801239, 0.9832988977432251, -0.038469646126031876, -0.7925114631652832, -1.7739111185073853, 0.44451895356178284, 0.25343701243400574, -1.720371127128601, -0.07771863043308258, -1.967751383781433, -1.5641752481460571, -0.13995113968849182, -0.7095766663551331, -0.028502946719527245, -1.3650153875350952, -1.5763801336288452, 1.0397465229034424, 0.49431195855140686, 0.671554684638977, 0.36000490188598633, 0.16796690225601196, -1.1087831258773804, 0.15539689362049103, -0.8622631430625916, 0.29417213797569275, 0.03951672092080116, -0.12424300611019135, -1.102290153503418, -0.3503955900669098, -2.297215223312378, -2.2174181938171387, 0.018135271966457367, -0.5109755396842957, 0.0834304690361023, 1.2299869060516357], [-0.45699167251586914, 0.1641208380460739, 0.9638965725898743, -0.8199802041053772, 0.6305345296859741, 0.21115437150001526, -0.6822673678398132, -0.10028962045907974, 0.00404271250590682, -0.06728005409240723, 0.025251153856515884, 0.7146804928779602, 1.2960739135742188, -1.417901635169983, 0.2877116799354553, -0.10402374714612961, 0.9810261130332947, -4.6195573806762695, -0.23427051305770874, -0.36257225275039673, -0.21829022467136383, 0.06934967637062073, 1.4699795246124268, -1.102140188217163, -0.09817743301391602, 0.20969925820827484, 0.2276223748922348, -2.963698625564575, 0.49393966794013977, -0.5428594946861267, 0.06837597489356995, 1.9720573425292969, -0.26261407136917114, -0.026406003162264824, 0.022516218945384026, 0.04809106886386871, -1.9853599071502686, 0.577430009841919, 0.20307284593582153, 0.09743686020374298, 0.6435931921005249, 0.13150367140769958, 0.5097997784614563, 2.290022611618042, -0.07992075383663177, -0.18759788572788239, 0.169054314494133, 1.4594472646713257], [-0.7204276323318481, -0.27526384592056274, -2.6687915325164795, -0.2550792098045349, -0.18093283474445343, 0.08100821077823639, -1.5275614261627197, -0.6128347516059875, -0.0077519346959888935, 2.063563585281372, 0.10190296918153763, -0.2911158502101898, 3.4280526638031006, -0.22585132718086243, -0.8237626552581787, 0.01977703906595707, -0.6730096936225891, 0.3678205907344818, -0.47896718978881836, -0.09437569230794907, -0.5609238147735596, 0.06321219354867935, 0.243332639336586, 0.48180726170539856, -0.15252114832401276, 1.2349659204483032, 0.1676240712404251, -0.03759392723441124, 1.3787083625793457, 0.9796445965766907, 0.07131365686655045, -0.9917008876800537, -0.19352644681930542, -0.6327372193336487, 0.5445359349250793, -0.09270403534173965, -1.4690488576889038, 0.12838131189346313, -0.3116045892238617, -0.010986526496708393, -0.24369168281555176, 0.27204811573028564, -0.384949266910553, 1.7877956628799438, -0.007117055356502533, 0.47685202956199646, -0.18866881728172302, -0.34415075182914734], [-0.4386059641838074, -0.06088470667600632, 1.1793041229248047, 0.4808356463909149, 0.44479888677597046, -0.6185466647148132, -0.02504875510931015, -2.1379311084747314, -0.010504232719540596, 0.6600564122200012, -0.03275534510612488, 0.7264462113380432, 0.2683725357055664, -0.44460999965667725, 0.2906419038772583, -0.20744970440864563, 0.39491721987724304, -0.6538973450660706, -0.3550007939338684, -0.4143410325050354, 0.7187291383743286, -0.19628508388996124, -0.44173213839530945, -0.5764961838722229, -0.24584633111953735, 0.22525708377361298, 0.26853862404823303, -2.1050527095794678, 0.7444223761558533, 0.9855399131774902, -0.5286116600036621, -1.039520263671875, -0.025001905858516693, -0.10657863318920135, -0.0708458423614502, 0.028821125626564026, -1.7344930171966553, -0.6896965503692627, 0.18175937235355377, 0.18910035490989685, 2.2162821292877197, 0.9416605830192566, 2.053434371948242, -1.5911657810211182, 0.03147536516189575, -0.3860221207141876, -0.46945297718048096, 0.8161872625350952]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(fused_add_rmsnorm, json.loads(_TESTS), 'cuda', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")